In [ ]:
import xarray as xr

# Datasets

In [ ]:
import json
from collections import defaultdict

with open(r"\\10.230.28.136\cez248024\Objective 1\Whiplash results\Cause-Effect\filtered_whiplash_events.json", "r") as f:
    loaded_dict3 = json.load(f)

In [ ]:
import json
from collections import defaultdict

with open(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Effect\filtered_whiplash_events_impact.json", "r") as f:
    loaded_dict2 = json.load(f)

In [ ]:
import json
from collections import defaultdict

with open(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause\filtered_whiplash_events_cause.json", "r") as f:
    loaded_dict1 = json.load(f)

In [ ]:
whiplash_dict_c_e = defaultdict(list, {eval(k): v for k, v in loaded_dict3.items()})

In [ ]:
whiplash_dict_e = defaultdict(list, {eval(k): v for k, v in loaded_dict2.items()})

In [ ]:
whiplash_dict_c = defaultdict(list, {eval(k): v for k, v in loaded_dict1.items()})

In [ ]:
whiplash_3_yearly_grids = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\whiplash_affected_grids_1980_2019.nc")

In [ ]:
whiplash_2_yearly_grids = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Effect\whiplash_affected_grids_1980_2019_impact.nc")

In [ ]:
whiplash_1_yearly_grids = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause\whiplash_affected_grids_1980_2019_cause.nc")

In [ ]:
grun=xr.open_dataset(r"\\10.230.28.136\cez248024\Objective 1\(FINAL) Hydroclimate variables used\G-RUN_ENSEMBLE_MMM.nc")

In [ ]:
# Rename coordinates
grun = grun.rename({"X": "lon", "Y": "lat"})

# Select time range 1980-01-01 to 2019-12-31
runoff= grun.sel(time=slice("1980-01-01", "2019-12-31"))

In [ ]:
nlat = grun.sizes['lat']
nlon = grun.sizes['lon']

lat_values = grun['lat'].values
lon_values = grun['lon'].values

# Spatial occurrence temporal trend

In [ ]:
import numpy as np

In [ ]:
filtered_avg_map1 = np.full((nlat, nlon), np.nan)
filtered_avg_map2 = np.full((nlat, nlon), np.nan)
filtered_avg_map3 = np.full((nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in whiplash_dict_e.items():
    scores = [ev["score"] for ev in ev_list]
    filtered_avg_map2[i, j] = np.mean(scores)

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():
    scores = [ev["score"] for ev in ev_list]
    filtered_avg_map3[i, j] = np.mean(scores)

In [ ]:
for (i, j), ev_list in whiplash_dict_c.items():
    scores = [ev["score"] for ev in ev_list]
    filtered_avg_map1[i, j] = np.mean(scores)

In [ ]:
whiplash_1_yearly_grids

In [ ]:
perc_pr = (whiplash_1_yearly_grids / 259200) * 100
perc_eff = (whiplash_2_yearly_grids / 259200) * 100
perc_comb = (whiplash_3_yearly_grids / 259200) * 100

In [ ]:
perc_pr['affected_grids'].max()

In [ ]:
perc_eff['affected_grids'].max()

In [ ]:
perc_comb['affected_grids'].max()

In [ ]:
# Example datasets
ds_pr = perc_pr      # precipitation definition
ds_eff = perc_eff# effect definition
ds_comb = perc_comb  # combined definition

years = whiplash_1_yearly_grids['year'].values

pr_vals = perc_pr['affected_grids'].values
eff_vals = perc_eff['affected_grids'].values
comb_vals = perc_comb['affected_grids'].values

In [ ]:
pr_vals

In [ ]:
import numpy as np

# Linear trends
coef_pr = np.polyfit(years, pr_vals, 1)
coef_eff = np.polyfit(years, eff_vals, 1)
coef_comb = np.polyfit(years, comb_vals, 1)

trend_pr = np.polyval(coef_pr, years)
trend_eff = np.polyval(coef_eff, years)
trend_comb = np.polyval(coef_comb, years)

In [ ]:
trend_pr_dec = coef_pr[0] * 10
trend_eff_dec = coef_eff[0] * 10
trend_comb_dec = coef_comb[0] * 10

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
# ---- Global style ----
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.linewidth": 1.2
})


# ---- Create figure ----
fig, ax = plt.subplots(figsize=(9,5))


ax.plot(
    years, pr_vals,
    color="#0072B2",
    linewidth=2.2,
    label=f"Cause definition ({trend_pr_dec:.2f} % decade$^{{-1}}$)"
)

ax.plot(
    years, eff_vals,
    color="#009E73",
    linewidth=2.2,
    label=f"Effect definition ({trend_eff_dec:.2f} % decade$^{{-1}}$)"
)

ax.plot(
    years, comb_vals,
    color="#D55E00",
    linewidth=2.2,
    label=f"Cause–Effect definition ({trend_comb_dec:.2f} % decade$^{{-1}}$)"
)

# Trend lines
ax.plot(
    years, trend_pr,
    color="#0072B2",
    linestyle="--",
    linewidth=1.6
)

ax.plot(
    years, trend_eff,
    color="#009E73",
    linestyle="--",
    linewidth=1.6
)

ax.plot(
    years, trend_comb,
    color="#D55E00",
    linestyle="--",
    linewidth=1.6
)
# ---- Axis formatting ----
ax.set_xlabel("Year", fontsize=13, fontweight="bold")
ax.set_ylabel("Affected grid cells (%)", fontsize=13, fontweight="bold")

ax.set_xlim(1980, 2020)
ax.set_ylim(0, 17)

ax.tick_params(
    axis='both',
    which='major',
    labelsize=11,
    width=1.1,
    length=5,
    direction="in"
)

# Minor ticks
ax.xaxis.set_minor_locator(mticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))


# ---- Grid (very subtle) ----
ax.grid(
    which='major',
    linestyle='--',
    linewidth=0.5,
    alpha=0.4
)


# ---- Legend ----
legend = ax.legend(
    frameon=False,
    fontsize=11,
    loc="upper left"
)

for text in legend.get_texts():
    text.set_fontweight('bold')


# ---- Tight layout ----
plt.tight_layout()


# ---- Save high resolution ----
plt.savefig(
    "whiplash_affected_grids_timeseries.tif",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

# ---- Global style ----
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14,
    "font.weight": "bold",
    "axes.linewidth": 1.2
})

# ---- Create figure ----
fig, ax = plt.subplots(figsize=(9,5))

# ---- Plot time series ----
ax.plot(years, pr_vals, color="#0072B2", linewidth=2.4)
ax.plot(years, eff_vals, color="#009E73", linewidth=2.4)
ax.plot(years, comb_vals, color="#D55E00", linewidth=2.4)

# ---- Trend lines ----
ax.plot(years, trend_pr, color="#0072B2", linestyle="--", linewidth=2)
ax.plot(years, trend_eff, color="#009E73", linestyle="--", linewidth=2)
ax.plot(years, trend_comb, color="#D55E00", linestyle="--", linewidth=2)

# ---- Axis formatting ----
ax.set_xlabel("Year", fontsize=14, fontweight="bold")
ax.set_ylabel("Affected grid cells (%)", fontsize=14, fontweight="bold")

ax.set_xlim(1979, 2020)
ax.set_ylim(0, 20)

ax.tick_params(
    axis='both',
    which='major',
    labelsize=14,
    width=1.2,
    length=6,
    direction="in"
)

# Minor ticks
ax.xaxis.set_minor_locator(mticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))

# ---- Grid ----
ax.grid(which='major', linestyle='--', linewidth=0.5, alpha=0.4)

# ---- Legend handles ----
handles_def = [
    Line2D([0],[0], color="#0072B2", lw=2.4, linestyle='-', label="Meteorological definition"),
    Line2D([0],[0], color="#009E73", lw=2.4, linestyle='-', label="Impact definition"),
    Line2D([0],[0], color="#D55E00", lw=2.4, linestyle='-', label="Integrated definition")
]

handles_trend = [
    Line2D([0],[0], color="#0072B2", lw=2, linestyle='--',
           label=f"Meteorological trend ({trend_pr_dec:.2f} % decade$^{{-1}}$)"),
    Line2D([0],[0], color="#009E73", lw=2, linestyle='--',
           label=f"Impact trend ({trend_eff_dec:.2f} % decade$^{{-1}}$)"),
    Line2D([0],[0], color="#D55E00", lw=2, linestyle='--',
           label=f"Integrated trend ({trend_comb_dec:.2f} % decade$^{{-1}}$)")
]

# ---- Legend 1 (definitions) ----
legend1 = ax.legend(
    handles=handles_def,
    loc="upper left",
    bbox_to_anchor=(0.01,0.99),
    frameon=True,
    fontsize=12,
    handlelength=3,
    labelspacing=0.6,
    borderpad=0.6
)

# ---- Legend 2 (trends) ----
legend2 = ax.legend(
    handles=handles_trend,
    loc="upper right",
    bbox_to_anchor=(0.99,0.99),
    frameon=True,
    fontsize=12,
    handlelength=3,
    labelspacing=0.6,
    borderpad=0.6
)

# Keep both legends
ax.add_artist(legend1)

# ---- Legend styling ----
for legend in [legend1, legend2]:
    legend.get_frame().set_edgecolor("black")
    legend.get_frame().set_linewidth(1.2)
    legend.get_frame().set_facecolor("white")
    for text in legend.get_texts():
        text.set_fontweight("bold")

# ---- Tight layout ----
plt.tight_layout()

# ---- Save figure ----
plt.savefig(
    "whiplash_affected_grids_timeseries.tif",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Average intensity across definitions

In [ ]:
filtered_avg_map1

In [ ]:
filtered_avg_map1da = xr.DataArray(
    filtered_avg_map1,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="avg_filtered_whiplash_score_whole"
)

In [ ]:
filtered_avg_map1da_land.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause\avg_intensity_c_e.nc")

In [ ]:
filtered_avg_map2da = xr.DataArray(
    filtered_avg_map2,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="avg_filtered_whiplash_score_whole"
)

In [ ]:
filtered_avg_map2da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Effect\avg_intensity_e.nc")

In [ ]:
filtered_avg_map3da = xr.DataArray(
    filtered_avg_map3,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="avg_filtered_whiplash_score_whole"
)

In [ ]:
filtered_avg_map3da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\avg_intensity_c_e.nc")

In [ ]:
import cartopy.io.shapereader as shpreader
import shapely.geometry as sgeom
import shapely.vectorized
import cartopy.feature as cfeature

land_feature = cfeature.NaturalEarthFeature(
    'physical', 'land', '110m'
)

land_geoms = list(land_feature.geometries())

import numpy as np

lon2d, lat2d = np.meshgrid(filtered_avg_map2da.lon, filtered_avg_map2da.lat)

mask = np.zeros(lon2d.shape, dtype=bool)

for geom in land_geoms:
    mask |= shapely.vectorized.contains(geom, lon2d, lat2d)

In [ ]:
filtered_avg_map1da_land = filtered_avg_map1da.where(mask)

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 14

fig = plt.figure(figsize=(14,3), dpi=600)

# Layout: 3 maps + vertical colorbar
gs = gridspec.GridSpec(
    1, 4,
    width_ratios=[1,1,1,0.05],
    wspace=0.15
)

ax1 = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[0,1], projection=ccrs.PlateCarree())
ax3 = fig.add_subplot(gs[0,2], projection=ccrs.PlateCarree())

# Shared normalization
norm = mcolors.Normalize(vmin=0, vmax=0.5)

cmap = "viridis"

# -------- Panel (a)
m1 = filtered_avg_map1da_land.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax1.set_title("(a) Meteorological Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (b)
m2 = filtered_avg_map2da.plot(
    ax=ax2,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax2.set_title("(b) Impact Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (c)
m3 = filtered_avg_map3da.plot(
    ax=ax3,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax3.set_title("(c) Integrated Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Map styling
for ax in [ax1, ax2, ax3]:

    ax.set_extent([-180,180,-90,90])

    ax.coastlines(resolution="110m", linewidth=0.6)

    ax.add_feature(
        cfeature.OCEAN,
        facecolor="lightgrey",
        edgecolor="none"
    )

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")
    
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
        spine.set_edgecolor("black")
# -------- Vertical colorbar
cax = fig.add_subplot(gs[0,3])

cbar = fig.colorbar(
    m3,
    cax=cax,
    orientation="vertical",
    extend="max"
)

cbar.set_label(
    "Average Whiplash Intensity",
    fontsize=12,
    fontweight="bold"
)

cbar.ax.tick_params(
    labelsize=12,
    direction="in",
    length=4,
    width=1
)

cbar.outline.set_linewidth(1.2)

plt.savefig(
    "Whiplash_Definition_Comparison.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

from matplotlib import ticker as mticker

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 14

fig = plt.figure(figsize=(14,3), dpi=600)

# Layout: 3 maps + vertical colorbar
gs = gridspec.GridSpec(
    1, 4,
    width_ratios=[1,1,1,0.05],
    wspace=0.15
)

ax1 = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[0,1], projection=ccrs.PlateCarree())
ax3 = fig.add_subplot(gs[0,2], projection=ccrs.PlateCarree())

# Shared normalization
norm = mcolors.Normalize(vmin=0, vmax=0.5)
cmap = "viridis"

# -------- Panel (a)
m1 = filtered_avg_map1da_land.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)
ax1.set_title("(a) Meteorological Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (b)
m2 = filtered_avg_map2da.plot(
    ax=ax2,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)
ax2.set_title("(b) Impact Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (c)
m3 = filtered_avg_map3da.plot(
    ax=ax3,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)
ax3.set_title("(c) Integrated Definition", fontweight="bold", fontsize=14, pad=6)

for ax in [ax1, ax2, ax3]:

    ax.set_extent([-180,180,-90,90])
    ax.coastlines(resolution="110m", linewidth=0.6)
    ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.5,
        color='black',
        alpha=0.5,
        linestyle='--',
        x_inline=False,
        y_inline=False
    )

    # Show tick numbers where desired
    gl.top_labels = True
    gl.bottom_labels = False
    gl.left_labels = False
    gl.right_labels = True

    # Format numbers only
    from matplotlib import ticker as mticker
    gl.xformatter = mticker.FormatStrFormatter('%d°')
    gl.yformatter = mticker.FormatStrFormatter('%d°')

    # Remove the axis label text (Cartopy 0.21+)
    gl.xlabel_style = {'visible': False}  # hides the "longitude" label
    gl.ylabel_style = {'visible': False}  # hides the "latitude" label

    ax.set_xticks([])
    ax.set_yticks([])

# -------- Vertical colorbar
cax = fig.add_subplot(gs[0,3])
cbar = fig.colorbar(
    m3,
    cax=cax,
    orientation="vertical",
    extend="max"
)
cbar.set_label("Average Whiplash Intensity", fontsize=12, fontweight="bold", family='Times New Roman')
cbar.ax.tick_params(labelsize=12, direction="in", length=4, width=1)
cbar.outline.set_linewidth(1.2)

#plt.savefig(
   #"Whiplash_Definition_Comparison.tiff",
    #dpi=600,
    #bbox_inches="tight"
#)
plt.show()

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

from matplotlib import ticker as mticker

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 14

fig = plt.figure(figsize=(21,3), dpi=600)

# Layout: 3 maps + vertical colorbar
gs = gridspec.GridSpec(
    1, 4,
    width_ratios=[1,1,1,0.05],
    wspace=0.15
)

ax1 = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[0,1], projection=ccrs.PlateCarree())
ax3 = fig.add_subplot(gs[0,2], projection=ccrs.PlateCarree())

# Shared normalization
norm = mcolors.Normalize(vmin=0, vmax=0.5)
cmap = "viridis"

# -------- Panel (a)
m1 = filtered_avg_map1da_land.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)
ax1.set_title("(a) Meteorological Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (b)
m2 = filtered_avg_map2da.plot(
    ax=ax2,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)
ax2.set_title("(b) Impact Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (c)
m3 = filtered_avg_map3da.plot(
    ax=ax3,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)
ax3.set_title("(c) Integrated Definition", fontweight="bold", fontsize=14, pad=6)

for ax in [ax1, ax2, ax3]:
    ax.set_extent([-180, 180, -90, 90])
    ax.coastlines(resolution="110m", linewidth=0.6)
    ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")
    
    # Draw gridlines but prevent Cartopy from adding 'Lat'/'Lon' labels
    gl = ax.gridlines(draw_labels=False, linewidth=0.5, color='black', linestyle='--', alpha=0.5)
    
    # Turn off the automatic "Lat" and "Lon" labels
    gl.top_labels = False# Turn off the automatic "Lat" and "Lon" labels
    gl.right_labels = False
    gl.bottom_labels = False
    gl.left_labels = False
    
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)   # increase thickness (try 2–3 for Nature-style)
        spine.set_edgecolor('black')
    # Set Matplotlib ticks
    xticks = np.arange(-180, 181, 60)
    yticks = np.arange(-90, 91, 30)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    
    # Manually set tick labels in degrees
    ax.set_xticklabels([f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks], fontsize=12)
    ax.set_yticklabels([f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks], fontsize=12)
    # ---- FORCE BOLD TICKS ----
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
# -------- Vertical colorbar
cax = fig.add_subplot(gs[0,3])
cbar = fig.colorbar(
    m3,
    cax=cax,
    orientation="vertical",
    extend="max"
)
cbar.set_label("Average Whiplash Intensity", fontsize=12, fontweight="bold", family='Times New Roman')
cbar.ax.tick_params(labelsize=14, direction="in", length=4, width=1)
cbar.outline.set_linewidth(1.2)

plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Whiplash_Definition_Comparison.tiff",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12

fig = plt.figure(figsize=(9,5), dpi=600)

gs = gridspec.GridSpec(
    2, 3,
    width_ratios=[1,1,0.05],   # last column = colorbar
    hspace=0.25,
    wspace=0.15
)

ax1 = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[0,1], projection=ccrs.PlateCarree())
ax3 = fig.add_subplot(gs[1,0:2], projection=ccrs.PlateCarree())

# colorbar axis
cax = fig.add_subplot(gs[:,2])

# Shared normalization
norm = mcolors.Normalize(vmin=0, vmax=0.5)

cmap = "viridis"   # color-blind friendly

# -------- Panel (a)
m1 = filtered_avg_map1da_land.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax1.set_title("(a) Cause Definition", fontweight="bold", pad=5)

# -------- Panel (b)
m2 = filtered_avg_map2da.plot(
    ax=ax2,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax2.set_title("(b) Effect Definition", fontweight="bold", pad=5)

# -------- Panel (c)
m3 = filtered_avg_map3da.plot(
    ax=ax3,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax3.set_title("(c) Cause–Effect Definition", fontweight="bold", pad=5)

# -------- Map styling
for ax in [ax1, ax2, ax3]:

    ax.set_extent([-180,180,-90,90])

    ax.coastlines(resolution="110m", linewidth=0.6)

    ax.add_feature(
        cfeature.OCEAN,
        facecolor="lightgrey",
        edgecolor="none"
    )

    #ax.add_feature(
        #cfeature.BORDERS,
        #linewidth=0.4,
        #edgecolor="black"
    #)

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

# colorbar axis
#cax = fig.add_subplot(gs[:,2])

cbar = fig.colorbar(
    m3,
    cax=cax,
    orientation="vertical",
    extend="max"
)

cbar.set_label(
    "Average Whiplash Intensity",
    fontsize=12,
    fontweight="bold"
)

cbar.ax.tick_params(
    labelsize=11,
    direction="in",
    length=4
)

plt.savefig(
    "Whiplash_Definition_Comparison.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Detected v/s validated

In [ ]:
det_val_c_e = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\detected_validated_c_e.nc")

In [ ]:
det_val_c_e

In [ ]:
det_val_e = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Effect\det_val_e.nc")

In [ ]:
det_val_c = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause\det_val_c.nc")

In [ ]:
det_val_c_da = det_val_c ['overlap']

In [ ]:
det_val_e_da = det_val_e ['overlap']

In [ ]:
det_val_c_e_da = det_val_c_e['overlap']

In [ ]:
det_val_c_da_land = det_val_c_da.where(mask)

In [ ]:
import numpy as np

In [ ]:
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# ----------------------
# Settings
# ----------------------
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 14

color_detected = "#0072B2"
color_validated = "#D55E00"

# ----------------------
# DataArrays
# ----------------------
da1 = det_val_c_da_land
da2 = det_val_e_da
da3 = det_val_c_e_da

datasets = [da1, da2, da3]

titles = [
    "(a) Meteorological Definition",
    "(b) Impact Definition",
    "(c) Integrated Definition"
]

# ----------------------
# Figure layout
# ----------------------
fig = plt.figure(figsize=(14,3), dpi=600)

gs = gridspec.GridSpec(
    2, 3,
    height_ratios=[1,0.08],
    wspace=0.12,
    hspace=0.05
)

axes = [fig.add_subplot(gs[0,i], projection=ccrs.PlateCarree()) for i in range(3)]

# ----------------------
# Plot maps
# ----------------------
for ax, da, title in zip(axes, datasets, titles):

    ax.set_extent([-180,180,-90,90])

    ax.coastlines(resolution="110m", linewidth=0.6)

    ax.add_feature(
        cfeature.OCEAN,
        facecolor="lightgrey",
        edgecolor="none"
    )

    # Extract coordinates and values
    lat_vals = da.lat.values
    lon_vals = da.lon.values
    vals = da.values

    # Create grid
    lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

    # Detected
    mask_detected = vals == 1

    ax.scatter(
        lon2d[mask_detected],
        lat2d[mask_detected],
        color=color_detected,
        s=0.5,
        transform=ccrs.PlateCarree()
    )

    # Detected + validated
    mask_validated = vals == 2

    ax.scatter(
        lon2d[mask_validated],
        lat2d[mask_validated],
        color=color_validated,
        s=0.5,
        transform=ccrs.PlateCarree()
    )

    ax.set_title(title, fontsize=14, fontweight="bold")

    ax.set_xticks([])
    ax.set_yticks([])

    # Border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
        spine.set_edgecolor("black")

# ----------------------
# Legend
# ----------------------
legend_ax = fig.add_subplot(gs[1,:])
legend_ax.axis("off")

legend_elements = [
    Line2D([0],[0], marker='o', color='w',
           markerfacecolor=color_detected,
           markersize=8,
           label="Detected"),

    Line2D([0],[0], marker='o', color='w',
           markerfacecolor=color_validated,
           markersize=8,
           label="Detected + Validated")
]

legend_ax.legend(
    handles=legend_elements,
    loc="center",
    ncol=2,
    frameon=False,
    fontsize=14
)

plt.tight_layout()

plt.savefig(
    "Whiplash_Detected_Validated_Comparison.tiff",
    dpi=50,
    bbox_inches="tight"
)

plt.show()

# Validation hit rate

In [ ]:
VHR_c_e = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\VHR_c_e.nc")
VHR_e = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Effect\VHR_e.nc")
VHR_c = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause\VHR_c.nc")

In [ ]:
VHR_c_da = VHR_c['VHR']
VHR_e_da = VHR_e['VHR']
VHR_c_e_da = VHR_c_e['VHR']

In [ ]:
#VHR_c_da_land = VHR_c_da.where(mask)

In [ ]:
VHR_c_da_land.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause\VHR_c_land.nc")

In [ ]:
VHR_c_da_land_ds = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause\VHR_c_land.nc")

In [ ]:
VHR_c_da_land = VHR_c_da_land_ds['VHR']

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 14

fig = plt.figure(figsize=(21, 3), dpi=600)

# Layout: 3 maps + vertical colorbar
gs = gridspec.GridSpec(
    1, 4,
    width_ratios=[1,1,1,0.05],
    wspace=0.15
)

ax1 = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[0,1], projection=ccrs.PlateCarree())
ax3 = fig.add_subplot(gs[0,2], projection=ccrs.PlateCarree())

# -------- Color scale for VHR
norm = mcolors.Normalize(vmin=0, vmax=100)

cmap = "magma"

# -------- Panel (a)
m1 = VHR_c_da_land.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax1.set_title("(a) Meteorological Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (b)
m2 =VHR_e_da.plot(
    ax=ax2,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax2.set_title("(b) Impact Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (c)
m3 = VHR_c_e_da.plot(
    ax=ax3,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax3.set_title("(c) Integrated Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Map styling
for ax in [ax1, ax2, ax3]:
    ax.set_extent([-180, 180, -90, 90])
    ax.coastlines(resolution="110m", linewidth=0.6)
    ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")
    
    # Draw gridlines but prevent Cartopy from adding 'Lat'/'Lon' labels
    gl = ax.gridlines(draw_labels=False, linewidth=0.5, color='black', linestyle='--', alpha=0.5)
    
    # Turn off the automatic "Lat" and "Lon" labels
    gl.top_labels = False# Turn off the automatic "Lat" and "Lon" labels
    gl.right_labels = False
    gl.bottom_labels = False
    gl.left_labels = False
    
    # Set Matplotlib ticks
    xticks = np.arange(-180, 181, 60)
    yticks = np.arange(-90, 91, 30)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    
    # Manually set tick labels in degrees
    ax.set_xticklabels([f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks], fontsize=12)
    ax.set_yticklabels([f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks], fontsize=12)

    # ---- FORCE BOLD TICKS ----
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.8)
        spine.set_edgecolor("black")

# -------- Vertical colorbar
cax = fig.add_subplot(gs[0,3])

cbar = fig.colorbar(
    m3,
    cax=cax,
    orientation="vertical"
)

cbar.set_label(
    "Validation Hit Rate (%)",
    fontsize=14,
    fontweight="bold"
)

cbar.ax.tick_params(
    labelsize=12,
    direction="in",
    length=4,
    width=1
)

# Make tick labels bold
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

cbar.outline.set_linewidth(1.2)

#plt.savefig(
    #r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\Validation_Hit_Rate_Definition_Comparison.tiff",
    #dpi=600,
    #bbox_inches="tight"
#)

plt.show()

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 14

fig = plt.figure(figsize=(17,8), dpi=600)
gs = gridspec.GridSpec(
    2, 3,
    width_ratios=[1,1,0.05],
    height_ratios=[1,1],
    wspace=0.12,
    hspace=0.18
)

# Top row
ax1 = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[0,1], projection=ccrs.PlateCarree())

# Bottom row (centered)
ax3 = fig.add_subplot(gs[1,0:2], projection=ccrs.PlateCarree())

# Shared colorbar
cax = fig.add_subplot(gs[:,2])

# -------- Color scale for VHR
norm = mcolors.Normalize(vmin=0, vmax=100)

cmap = "magma"

# -------- Panel (a)
m1 = VHR_c_da_land.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax1.set_title("(a) Meteorological Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (b)
m2 =VHR_e_da.plot(
    ax=ax2,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax2.set_title("(b) Impact Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (c)
m3 = VHR_c_e_da.plot(
    ax=ax3,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax3.set_title("(c) Integrated Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Map styling
for ax in [ax1, ax2, ax3]:
    ax.set_extent([-180, 180, -90, 90])
    ax.coastlines(resolution="110m", linewidth=0.6)
    ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")
    
    # Draw gridlines but prevent Cartopy from adding 'Lat'/'Lon' labels
    gl = ax.gridlines(draw_labels=False, linewidth=0.5, color='black', linestyle='--', alpha=0.5)
    
    # Turn off the automatic "Lat" and "Lon" labels
    gl.top_labels = False# Turn off the automatic "Lat" and "Lon" labels
    gl.right_labels = False
    gl.bottom_labels = False
    gl.left_labels = False
    
    # Set Matplotlib ticks
    xticks = np.arange(-180, 181, 60)
    yticks = np.arange(-90, 91, 30)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    
    # Manually set tick labels in degrees
    ax.set_xticklabels([f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks], fontsize=12)
    ax.set_yticklabels([f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks], fontsize=12)

    # ---- FORCE BOLD TICKS ----
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.8)
        spine.set_edgecolor("black")

# -------- Vertical colorbar
#cax = fig.add_subplot(gs[:,2])

cbar = fig.colorbar(
    m3,
    cax=cax,
    orientation="vertical",
    ticks=np.arange(0, 101, 20)
)
cbar.ax.minorticks_off()
cbar.set_label(
    "Validation Hit Rate (%)",
    fontsize=14,
    fontweight="bold"
)

cbar.ax.tick_params(
    labelsize=12,
    direction="in",
    length=4,
    width=1
)

# Make tick labels bold
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

cbar.outline.set_linewidth(1.2)

plt.savefig(
    r"D:\IITD\Doctoral Thesis\Work and Progress ppts\nature communications\Plots_final\Validation_Hit_Rate_Definition_Comparison_2_by_3.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 14

fig = plt.figure(figsize=(9,14), dpi=600)

gs = gridspec.GridSpec(
    3, 2,
    width_ratios=[1, 0.045],
    height_ratios=[1,1,1],
    wspace=0.10,
    hspace=0.22
)

# -------- Map axes --------
ax1 = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[1,0], projection=ccrs.PlateCarree())
ax3 = fig.add_subplot(gs[2,0], projection=ccrs.PlateCarree())

# -------- Shared colorbar --------
cax = fig.add_subplot(gs[:,1])

# -------- Color scale for VHR
norm = mcolors.Normalize(vmin=0, vmax=100)

cmap = "magma"

# -------- Panel (a)
m1 = VHR_c_da_land.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax1.set_title("(a) Meteorological Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (b)
m2 =VHR_e_da.plot(
    ax=ax2,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax2.set_title("(b) Impact Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Panel (c)
m3 = VHR_c_e_da.plot(
    ax=ax3,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax3.set_title("(c) Integrated Definition", fontweight="bold", fontsize=14, pad=6)

# -------- Map styling
for ax in [ax1, ax2, ax3]:
    ax.set_extent([-180, 180, -90, 90])
    ax.coastlines(resolution="110m", linewidth=0.6)
    ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")
    
    # Draw gridlines but prevent Cartopy from adding 'Lat'/'Lon' labels
    gl = ax.gridlines(draw_labels=False, linewidth=0.5, color='black', linestyle='--', alpha=0.5)
    
    # Turn off the automatic "Lat" and "Lon" labels
    gl.top_labels = False# Turn off the automatic "Lat" and "Lon" labels
    gl.right_labels = False
    gl.bottom_labels = False
    gl.left_labels = False
    
    # Set Matplotlib ticks
    xticks = np.arange(-180, 181, 60)
    yticks = np.arange(-90, 91, 30)
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)
    
    # Manually set tick labels in degrees
    ax.set_xticklabels([f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks], fontsize=12)
    ax.set_yticklabels([f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks], fontsize=12)

    # ---- FORCE BOLD TICKS ----
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.8)
        spine.set_edgecolor("black")

# -------- Vertical colorbar
#cax = fig.add_subplot(gs[:,2])

cbar = fig.colorbar(
    m3,
    cax=cax,
    orientation="vertical",
    ticks=np.arange(0, 101, 20)
)
cbar.ax.minorticks_off()
cbar.set_label(
    "Validation Hit Rate (%)",
    fontsize=14,
    fontweight="bold"
)

cbar.ax.tick_params(
    labelsize=12,
    direction="in",
    length=4,
    width=1
)

# Make tick labels bold
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

cbar.outline.set_linewidth(1.2)

plt.savefig(
    r"D:\IITD\Doctoral Thesis\Work and Progress ppts\nature communications\Plots_final\Validation_Hit_Rate_Definition_Comparison_1_by_3.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


# Decadal spatial whiplash characteristics

In [ ]:
import numpy as np
import pandas as pd

decades = {
    "1980-1989": (1980, 1989),
    "1990-1999": (1990, 1999),
    "2000-2009": (2000, 2009),
    "2010-2019": (2010, 2019)
}

In [ ]:
decadal_stats = {d: {} for d in decades}

for (lat_i, lon_i), events in whiplash_dict_c_e.items():
    
    for decade, (start, end) in decades.items():
        
        freq = 0
        intensities = []
        durations = []
        
        for e in events:
            year = pd.to_datetime(e["tc_time"]).year
            
            if start <= year <= end:
                freq += 1
                intensities.append(e["score"])
                
                tc = pd.to_datetime(e["tc_time"])
                td = pd.to_datetime(e["td_time"])
                durations.append((td.year - tc.year)*12 + (td.month - tc.month) + 1)
        
        decadal_stats[decade][(lat_i, lon_i)] = {
            "frequency": freq,
            "intensity": np.mean(intensities) if intensities else np.nan,
            "duration": np.mean(durations) if durations else np.nan
        }

In [ ]:
decadal_stats

In [ ]:
decadal_max_stats = {d: {} for d in decades}

for (lat_i, lon_i), events in whiplash_dict_c_e.items():
    
    for decade, (start, end) in decades.items():
        
        freq = 0
        intensities = []
        durations = []
        
        for e in events:
            year = pd.to_datetime(e["tc_time"]).year
            
            if start <= year <= end:
                freq += 1
                intensities.append(e["score"])
                
                tc = pd.to_datetime(e["tc_time"])
                td = pd.to_datetime(e["td_time"])
                durations.append((td.year - tc.year)*12 + (td.month - tc.month) + 1)
        
        decadal_max_stats[decade][(lat_i, lon_i)] = {
            "frequency": freq,
            "intensity": np.max(intensities) if intensities else np.nan,
            "duration": np.max(durations) if durations else np.nan
        }

In [ ]:
decadal_max_maps = {}

for decade in decades:
    
    freq_max_map = np.full((nlat, nlon), np.nan)   # initialize with NaN
    intensity_max_map = np.full((nlat, nlon), np.nan)
    duration_max_map = np.full((nlat, nlon), np.nan)
    
    for (lat_i, lon_i), stats in decadal_max_stats[decade].items():
        
        if stats["frequency"] > 0:   # only keep grids with events
            
            freq_max_map[lat_i, lon_i] = stats["frequency"]
            intensity_max_map[lat_i, lon_i] = stats["intensity"]
            duration_max_map[lat_i, lon_i] = stats["duration"]
    
    decadal_max_maps[decade] = {
        "frequency": freq_max_map,
        "intensity": intensity_max_map,
        "duration": duration_max_map
    }

In [ ]:
decadal_avg_maps = {}

for decade in decades:
    
    freq_avg_map = np.full((nlat, nlon), np.nan)   # initialize with NaN
    intensity_avg_map = np.full((nlat, nlon), np.nan)
    duration_avg_map = np.full((nlat, nlon), np.nan)
    
    for (lat_i, lon_i), stats in decadal_stats[decade].items():
        
        if stats["frequency"] > 0:   # only keep grids with events
            
            freq_avg_map[lat_i, lon_i] = stats["frequency"]
            intensity_avg_map[lat_i, lon_i] = stats["intensity"]
            duration_avg_map[lat_i, lon_i] = stats["duration"]
    
    decadal_avg_maps[decade] = {
        "frequency": freq_avg_map,
        "intensity": intensity_avg_map,
        "duration": duration_avg_map
    }

In [ ]:
import numpy as np

min_duration = np.nanmin(decadal_maps[decade]['duration'])
print(min_duration)

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12

fig = plt.figure(figsize=(32,12), dpi=600)

# 3 rows (metrics) × 5 columns (4 maps + colorbar)
gs = gridspec.GridSpec(
    3, 5,
    width_ratios=[1,1,1,1,0.05],
    hspace=0.15,
    wspace=0.05
)

norm_int = mcolors.Normalize(vmin=0, vmax=0.5)   # adjust
norm_dur = mcolors.Normalize(vmin=4, vmax=10)  # adjust
norm_freq = mcolors.Normalize(vmin=0, vmax=3)  # adjust

cmap_int = "viridis"
cmap_dur = "cividis"
cmap_freq = "plasma"

axes = []

for r in range(3):
    row = []
    for c in range(4):
        ax = fig.add_subplot(gs[r,c], projection=ccrs.PlateCarree())
        row.append(ax)
    axes.append(row)

decades = ["1980-1989","1990-1999","2000-2009","2010-2019"]

for i, dec in enumerate(decades):

    m = axes[0][i].pcolormesh(
        lon_values,
        lat_values,
        decadal_avg_maps[dec]["intensity"],
        cmap=cmap_int,
        norm=norm_int,
        shading="auto",
        edgecolors="face",
        transform=ccrs.PlateCarree()
    )

    axes[0][i].set_title(dec, fontweight="bold")

for i, dec in enumerate(decades):

    m2 = axes[1][i].pcolormesh(
        lon_values,
        lat_values,
        decadal_maps[dec]["duration"],
        cmap=cmap_dur,
        norm=norm_dur,
        shading="auto",
        edgecolors="face",
        transform=ccrs.PlateCarree()
    )

for i, dec in enumerate(decades):

    m3 = axes[2][i].pcolormesh(
        lon_values,
        lat_values,
        decadal_maps[dec]["frequency"],
        cmap=cmap_freq,
        norm=norm_freq,
        shading="auto",
        edgecolors="face",
        transform=ccrs.PlateCarree()
    )

lon2d, lat2d = np.meshgrid(lon_values, lat_values)


for row in axes:
    for ax in row:
        ax.set_extent([-180, 180, -90, 90])
        ax.coastlines(resolution="110m", linewidth=0.6)
        ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")
    
        # Draw gridlines but prevent Cartopy from adding 'Lat'/'Lon' labels
        gl = ax.gridlines(draw_labels=False, linewidth=0.5, color='black', linestyle='--', alpha=0.5)
    
        # Turn off the automatic "Lat" and "Lon" labels
        gl.top_labels = False# Turn off the automatic "Lat" and "Lon" labels
        gl.right_labels = False
        gl.bottom_labels = False
        gl.left_labels = False
    
        # Set Matplotlib ticks
        xticks = np.arange(-180, 181, 60)
        yticks = np.arange(-90, 91, 30)
        ax.set_xticks(xticks)
        ax.set_yticks(yticks)
    
        # Manually set tick labels in degrees
        ax.set_xticklabels([f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks], fontsize=12)
        ax.set_yticklabels([f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks], fontsize=12)
        
        # ---- FORCE BOLD TICKS ----
        for label in ax.get_xticklabels():
            label.set_fontweight('bold')

        for label in ax.get_yticklabels():
            label.set_fontweight('bold')

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.8)

cax1 = fig.add_subplot(gs[0,4])

cbar1 = fig.colorbar(
    m,
    cax=cax1,
    orientation="vertical",
    shrink=0.7
)

cbar1.set_label(
    "Intensity",
    fontsize=14,
    fontweight="bold",
    labelpad=9
)


cax2 = fig.add_subplot(gs[1,4])

cbar2 = fig.colorbar(
    m2,
    cax=cax2,
    orientation="vertical",
    shrink=0.7
)

cbar2.set_label(
    "Duration (months)",
    fontsize=14,
    fontweight="bold",
    labelpad=9
)


cax3 = fig.add_subplot(gs[2,4])

cbar3 = fig.colorbar(
    m3,
    cax=cax3,
    orientation="vertical",
    shrink=0.7
)

cbar3.set_label(
    "Frequency (events/decade)",
    fontsize=14,
    fontweight="bold",
    labelpad=9
)
cbar1.ax.yaxis.set_label_coords(3.5, 0.5)
cbar2.ax.yaxis.set_label_coords(3.5, 0.5)
cbar3.ax.yaxis.set_label_coords(3.5, 0.5)

# ---- FORMAT ALL COLORBARS ----
for cbar in [cbar1, cbar2, cbar3]:
    cbar.ax.tick_params(labelsize=12, width=1.2)
    
    # Bold tick labels
    for label in cbar.ax.get_yticklabels():
        label.set_fontweight('bold')
    
    # Thicker outline
    cbar.outline.set_linewidth(1.5)

plt.tight_layout()
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\Decadal_IDF_integrated.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12

fig = plt.figure(figsize=(20,15), dpi=600)

gs = gridspec.GridSpec(
    3, 3,
    width_ratios=[1,1,0.05],
    hspace=0.15,
    wspace=0.05
)

norm_int = mcolors.Normalize(vmin=0, vmax=0.5)   # adjust
norm_dur = mcolors.Normalize(vmin=4, vmax=10)  # adjust
norm_freq = mcolors.Normalize(vmin=0, vmax=3)  # adjust

cmap_int = "viridis"
cmap_dur = "cividis"
cmap_freq = "plasma"

axes = []

for r in range(3):
    row = []
    for c in range(2):
        ax = fig.add_subplot(gs[r,c], projection=ccrs.PlateCarree())
        row.append(ax)
    axes.append(row)

decades = ["1980-1989","2010-2019"]

for i, dec in enumerate(decades):

    m = axes[0][i].pcolormesh(
        lon_values,
        lat_values,
        decadal_avg_maps[dec]["intensity"],
        cmap=cmap_int,
        norm=norm_int,
        shading="auto",
        edgecolors="face",
        transform=ccrs.PlateCarree()
    )

    axes[0][i].set_title(
    f"({chr(97+i)}) {dec} (Average Whiplash Intensity)",
    fontsize=14,
    fontweight="bold",
    pad=8
    )

for i, dec in enumerate(decades):

    m2 = axes[1][i].pcolormesh(
        lon_values,
        lat_values,
        decadal_avg_maps[dec]["duration"],
        cmap=cmap_dur,
        norm=norm_dur,
        shading="auto",
        edgecolors="face",
        transform=ccrs.PlateCarree()
    )
    axes[1][i].set_title(
    f"({chr(99+i)}) {dec} (Average Whiplash Duration)",
    fontsize=14,
    fontweight="bold",
    pad=8
    )

for i, dec in enumerate(decades):

    m3 = axes[2][i].pcolormesh(
        lon_values,
        lat_values,
        decadal_avg_maps[dec]["frequency"],
        cmap=cmap_freq,
        norm=norm_freq,
        shading="auto",
        edgecolors="face",
        transform=ccrs.PlateCarree()
    )
    axes[2][i].set_title(
    f"({chr(101+i)}) {dec} (Whiplash Frequency)",
    fontsize=14,
    fontweight="bold",
    pad=8)

lon2d, lat2d = np.meshgrid(lon_values, lat_values)


for row in axes:
    for ax in row:
        ax.set_extent([-180, 180, -90, 90])
        ax.coastlines(resolution="110m", linewidth=0.6)
        ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")
    
        # Draw gridlines but prevent Cartopy from adding 'Lat'/'Lon' labels
        gl = ax.gridlines(draw_labels=False, linewidth=0.5, color='black', linestyle='--', alpha=0.5)
    
        # Turn off the automatic "Lat" and "Lon" labels
        gl.top_labels = False# Turn off the automatic "Lat" and "Lon" labels
        gl.right_labels = False
        gl.bottom_labels = False
        gl.left_labels = False
    
        # Set Matplotlib ticks
        xticks = np.arange(-180, 181, 60)
        yticks = np.arange(-90, 91, 30)
        ax.set_xticks(xticks)
        ax.set_yticks(yticks)
    
        # Manually set tick labels in degrees
        ax.set_xticklabels([f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks], fontsize=12)
        ax.set_yticklabels([f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks], fontsize=12)
        
        # ---- FORCE BOLD TICKS ----
        for label in ax.get_xticklabels():
            label.set_fontweight('bold')

        for label in ax.get_yticklabels():
            label.set_fontweight('bold')

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.8)


cax1 = fig.add_subplot(gs[0,2])

cbar1 = fig.colorbar(
    m,
    cax=cax1,
    orientation="vertical",
    shrink=0.7
)

cbar1.set_label(
    "Intensity",
    fontsize=14,
    fontweight="bold",
    labelpad=9
)


cax2 = fig.add_subplot(gs[1,2])

cbar2 = fig.colorbar(
    m2,
    cax=cax2,
    orientation="vertical",
    shrink=0.7
)

cbar2.set_label(
    "Duration (months)",
    fontsize=14,
    fontweight="bold",
    labelpad=9
)


cax3 = fig.add_subplot(gs[2,2])

cbar3 = fig.colorbar(
    m3,
    cax=cax3,
    orientation="vertical",
    shrink=0.7
)

cbar3.set_label(
    "Frequency (events/decade)",
    fontsize=14,
    fontweight="bold",
    labelpad=9
)
cbar1.ax.yaxis.set_label_coords(2, 0.5)
cbar2.ax.yaxis.set_label_coords(2, 0.5)
cbar3.ax.yaxis.set_label_coords(2, 0.5)

# ---- FORMAT ALL COLORBARS ----
for cbar in [cbar1, cbar2, cbar3]:
    cbar.ax.tick_params(labelsize=12, width=1.2)
    
    # Bold tick labels
    for label in cbar.ax.get_yticklabels():
        label.set_fontweight('bold')
    
    # Thicker outline
    cbar.outline.set_linewidth(1.5)

plt.tight_layout()
plt.savefig(
    r"D:\IITD\Doctoral Thesis\Work and Progress ppts\nature communications\Plots_final\Supplementary\Decadal_IDF_avg_1980_2010.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12

fig = plt.figure(figsize=(20,15), dpi=600)

gs = gridspec.GridSpec(
    3, 3,
    width_ratios=[1,1,0.05],
    hspace=0.15,
    wspace=0.05
)

norm_int = mcolors.Normalize(vmin=0, vmax=0.5)   # adjust
norm_dur = mcolors.Normalize(vmin=4, vmax=10)  # adjust
norm_freq = mcolors.Normalize(vmin=0, vmax=3)  # adjust

cmap_int = "viridis"
cmap_dur = "cividis"
cmap_freq = "plasma"

axes = []

for r in range(3):
    row = []
    for c in range(2):
        ax = fig.add_subplot(gs[r,c], projection=ccrs.PlateCarree())
        row.append(ax)
    axes.append(row)

decades = ["1980-1989","2010-2019"]

for i, dec in enumerate(decades):

    m = axes[0][i].pcolormesh(
        lon_values,
        lat_values,
        decadal_max_maps[dec]["intensity"],
        cmap=cmap_int,
        norm=norm_int,
        shading="auto",
        edgecolors="face",
        transform=ccrs.PlateCarree()
    )

    axes[0][i].set_title(
    f"({chr(97+i)}) {dec} (Maximum Whiplash Intensity)",
    fontsize=14,
    fontweight="bold",
    pad=8
    )

for i, dec in enumerate(decades):

    m2 = axes[1][i].pcolormesh(
        lon_values,
        lat_values,
        decadal_max_maps[dec]["duration"],
        cmap=cmap_dur,
        norm=norm_dur,
        shading="auto",
        edgecolors="face",
        transform=ccrs.PlateCarree()
    )
    axes[1][i].set_title(
    f"({chr(99+i)}) {dec} (Maximum Whiplash Duration)",
    fontsize=14,
    fontweight="bold",
    pad=8
    )

for i, dec in enumerate(decades):

    m3 = axes[2][i].pcolormesh(
        lon_values,
        lat_values,
        decadal_max_maps[dec]["frequency"],
        cmap=cmap_freq,
        norm=norm_freq,
        shading="auto",
        edgecolors="face",
        transform=ccrs.PlateCarree()
    )
    axes[2][i].set_title(
    f"({chr(101+i)}) {dec} (Whiplash Frequency)",
    fontsize=14,
    fontweight="bold",
    pad=8)

lon2d, lat2d = np.meshgrid(lon_values, lat_values)


for row in axes:
    for ax in row:
        ax.set_extent([-180, 180, -90, 90])
        ax.coastlines(resolution="110m", linewidth=0.6)
        ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")
    
        # Draw gridlines but prevent Cartopy from adding 'Lat'/'Lon' labels
        gl = ax.gridlines(draw_labels=False, linewidth=0.5, color='black', linestyle='--', alpha=0.5)
    
        # Turn off the automatic "Lat" and "Lon" labels
        gl.top_labels = False# Turn off the automatic "Lat" and "Lon" labels
        gl.right_labels = False
        gl.bottom_labels = False
        gl.left_labels = False
    
        # Set Matplotlib ticks
        xticks = np.arange(-180, 181, 60)
        yticks = np.arange(-90, 91, 30)
        ax.set_xticks(xticks)
        ax.set_yticks(yticks)
    
        # Manually set tick labels in degrees
        ax.set_xticklabels([f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks], fontsize=12)
        ax.set_yticklabels([f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks], fontsize=12)
        
        # ---- FORCE BOLD TICKS ----
        for label in ax.get_xticklabels():
            label.set_fontweight('bold')

        for label in ax.get_yticklabels():
            label.set_fontweight('bold')

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.8)


cax1 = fig.add_subplot(gs[0,2])

cbar1 = fig.colorbar(
    m,
    cax=cax1,
    orientation="vertical",
    shrink=0.7
)

cbar1.set_label(
    "Intensity",
    fontsize=14,
    fontweight="bold",
    labelpad=9
)


cax2 = fig.add_subplot(gs[1,2])

cbar2 = fig.colorbar(
    m2,
    cax=cax2,
    orientation="vertical",
    shrink=0.7
)

cbar2.set_label(
    "Duration (months)",
    fontsize=14,
    fontweight="bold",
    labelpad=9
)


cax3 = fig.add_subplot(gs[2,2])

cbar3 = fig.colorbar(
    m3,
    cax=cax3,
    orientation="vertical",
    shrink=0.7
)

cbar3.set_label(
    "Frequency (events/decade)",
    fontsize=14,
    fontweight="bold",
    labelpad=9
)
cbar1.ax.yaxis.set_label_coords(2, 0.5)
cbar2.ax.yaxis.set_label_coords(2, 0.5)
cbar3.ax.yaxis.set_label_coords(2, 0.5)

# ---- FORMAT ALL COLORBARS ----
for cbar in [cbar1, cbar2, cbar3]:
    cbar.ax.tick_params(labelsize=12, width=1.2)
    
    # Bold tick labels
    for label in cbar.ax.get_yticklabels():
        label.set_fontweight('bold')
    
    # Thicker outline
    cbar.outline.set_linewidth(1.5)

plt.tight_layout()
plt.savefig(
    r"D:\IITD\Doctoral Thesis\Work and Progress ppts\nature communications\Plots_final\Supplementary\Decadal_IDF_max_1980_2010.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd

# Your objects
events = whiplash_dict_c_e
stats = decadal_stats

# --------------------------------------------------
# 1. Create coordinate lookup from the event dictionary
# --------------------------------------------------
coord_lookup = {}

for grid_key, event_list in events.items():
    if event_list:
        coord_lookup[grid_key] = {
            "lat": event_list[0]["lat"],
            "lon": event_list[0]["lon"]
        }

# --------------------------------------------------
# 2. Convert decadal statistics to a dataframe
# --------------------------------------------------
records = []

for decade, grid_stats in stats.items():
    for grid_key, values in grid_stats.items():

        if grid_key not in coord_lookup:
            continue

        records.append({
            "decade": decade,
            "lat_index": grid_key[0],
            "lon_index": grid_key[1],
            "lat": coord_lookup[grid_key]["lat"],
            "lon": coord_lookup[grid_key]["lon"],
            "frequency": values.get("frequency", np.nan),
            "intensity": values.get("intensity", np.nan),
            "duration": values.get("duration", np.nan)
        })

decadal_df = pd.DataFrame(records)

# Ensure numeric columns
for col in ["lat", "lon", "frequency", "intensity"]:
    decadal_df[col] = pd.to_numeric(
        decadal_df[col],
        errors="coerce"
    )

In [ ]:
regions = {
    "North America": {
        "lat_min": 15, "lat_max": 75,
        "lon_min": -170, "lon_max": -50
    },
    "South America": {
        "lat_min": -56, "lat_max": 13,
        "lon_min": -82, "lon_max": -34
    },
    "Europe": {
        "lat_min": 35, "lat_max": 72,
        "lon_min": -12, "lon_max": 45
    },
    "Central Asia": {
        "lat_min": 35, "lat_max": 55,
        "lon_min": 45, "lon_max": 90
    },
    "East Asia": {
        "lat_min": 20, "lat_max": 55,
        "lon_min": 90, "lon_max": 145
    },
    "South Asia": {
        "lat_min": 5, "lat_max": 35,
        "lon_min": 60, "lon_max": 100
    },
    "Southeast Asia": {
        "lat_min": -11, "lat_max": 24,
        "lon_min": 95, "lon_max": 141
    },
    "Middle East": {
        "lat_min": 12, "lat_max": 42,
        "lon_min": 25, "lon_max": 65
    },
    "Africa": {
        "lat_min": -35, "lat_max": 37,
        "lon_min": -20, "lon_max": 52
    },
    "Australia": {
        "lat_min": -44, "lat_max": -10,
        "lon_min": 112, "lon_max": 154
    }
}

In [ ]:
def percent_change(old, new):
    if pd.isna(old) or old == 0:
        return np.nan

    return ((new - old) / old) * 100


def regional_decadal_change(
    df,
    region_bounds,
    decade_early="1980-1989",
    decade_late="2010-2019"
):
    # Select region
    regional = df.loc[
        df["lat"].between(
            region_bounds["lat_min"],
            region_bounds["lat_max"]
        )
        &
        df["lon"].between(
            region_bounds["lon_min"],
            region_bounds["lon_max"]
        )
    ].copy()

    early = regional.loc[
        regional["decade"].eq(decade_early)
    ]

    late = regional.loc[
        regional["decade"].eq(decade_late)
    ]

    # Only affected cells
    early_affected = early.loc[
        early["frequency"].gt(0)
    ]

    late_affected = late.loc[
        late["frequency"].gt(0)
    ]

    # Spatial extent
    n_early = len(early_affected)
    n_late = len(late_affected)

    # Mean metrics across affected cells
    intensity_early = early_affected["intensity"].mean()
    intensity_late = late_affected["intensity"].mean()

    frequency_early = early_affected["frequency"].mean()
    frequency_late = late_affected["frequency"].mean()

    duration_early = early_affected["duration"].mean()
    duration_late = late_affected["duration"].mean()

    return {
        "affected_cells_1980s": n_early,
        "affected_cells_2010s": n_late,

        "extent_change_pct": percent_change(
            n_early, n_late
        ),

        "mean_intensity_1980s": intensity_early,
        "mean_intensity_2010s": intensity_late,
        "intensity_change_pct": percent_change(
            intensity_early, intensity_late
        ),

        "mean_frequency_1980s": frequency_early,
        "mean_frequency_2010s": frequency_late,
        "frequency_change_pct": percent_change(
            frequency_early, frequency_late
        ),

        "mean_duration_1980s": duration_early,
        "mean_duration_2010s": duration_late,
        "duration_change_pct": percent_change(
            duration_early, duration_late
        )
    }
results = []

for region_name, bounds in regions.items():

    result = regional_decadal_change(
        decadal_df,
        bounds
    )

    result["region"] = region_name
    results.append(result)

regional_changes = (
    pd.DataFrame(results)
    .set_index("region")
)

change_table = regional_changes[
    [
        "extent_change_pct",
        "intensity_change_pct",
        "frequency_change_pct",
        "duration_change_pct"
    ]
]

change_table.round(2)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib import rcParams


# ============================================================
# 1. CONVERT ONE DECADE FROM DICTIONARY TO ARRAYS
# ============================================================

def decade_to_arrays(decadal_stats, decade, nlat, nlon):

    intensity = np.full((nlat, nlon), np.nan)
    duration  = np.full((nlat, nlon), np.nan)

    # Frequency should be zero where there were no events
    frequency = np.zeros((nlat, nlon), dtype=float)

    for (i, j), stats in decadal_stats[decade].items():

        intensity[i, j] = stats["intensity"]
        duration[i, j]  = stats["duration"]
        frequency[i, j] = stats["frequency"]

    return intensity, duration, frequency

In [ ]:
i80, d80, f80 = decade_to_arrays(
    decadal_stats,
    "1980-1989",
    nlat,
    nlon
)

i90, d90, f90 = decade_to_arrays(
    decadal_stats,
    "1990-1999",
    nlat,
    nlon
)

i00, d00, f00 = decade_to_arrays(
    decadal_stats,
    "2000-2009",
    nlat,
    nlon
)

i10, d10, f10 = decade_to_arrays(
    decadal_stats,
    "2010-2019",
    nlat,
    nlon
)

#### first v/s last decade

In [ ]:
# Replace NaN = no detected whiplash in that decade with zero
i80_zero = np.nan_to_num(i80, nan=0.0)
i10_zero = np.nan_to_num(i10, nan=0.0)

d80_zero = np.nan_to_num(d80, nan=0.0)
d10_zero = np.nan_to_num(d10, nan=0.0)

# Difference: 2010s minus 1980s
diff_int_80_10 = i10_zero - i80_zero
diff_dur_80_10 = d10_zero - d80_zero

# Frequency already has meaningful zeros
diff_freq_80_10 = f10 - f80

In [ ]:
def make_da(array, name):

    return xr.DataArray(
        array,
        coords={
            "lat": lat_values,
            "lon": lon_values
        },
        dims=["lat", "lon"],
        name=name
    )


# 1980s vs 2010s

diff_int_80_10_da = make_da(
    diff_int_80_10,
    "intensity_change"
)

diff_dur_80_10_da = make_da(
    diff_dur_80_10,
    "duration_change"
)

diff_freq_80_10_da = make_da(
    diff_freq_80_10,
    "frequency_change"
)

In [ ]:
for name, da in {
    "Intensity 80s-10s": diff_int_80_10_da,
    "Duration 80s-10s": diff_dur_80_10_da,
    "Frequency 80s-10s": diff_freq_80_10_da
}.items():

    print(
        name,
        "min =", float(da.min(skipna=True)),
        "max =", float(da.max(skipna=True)),
        "mean =", float(da.mean(skipna=True))
    )

In [ ]:
def plot_difference_maps(
    intensity_da,
    duration_da,
    frequency_da,
    save_path=None
):

    rcParams["font.family"] = "Times New Roman"
    rcParams["font.size"] = 12

    fig = plt.figure(
        figsize=(11.5, 7),
        dpi=600
    )

    # ------------------------------------------------
    # 2 rows × 4 columns
    # top: two maps
    # bottom: one centred map
    # ------------------------------------------------
    gs = fig.add_gridspec(
        2, 4,
        wspace=0.25,
        hspace=0.28
    )

    ax1 = fig.add_subplot(
        gs[0, 0:2],
        projection=ccrs.PlateCarree()
    )

    ax2 = fig.add_subplot(
        gs[0, 2:4],
        projection=ccrs.PlateCarree()
    )

    ax3 = fig.add_subplot(
        gs[1, 1:3],
        projection=ccrs.PlateCarree()
    )

    axes = [ax1, ax2, ax3]

    datasets = [
        intensity_da,
        duration_da,
        frequency_da
    ]

    titles = [
        "(a) Change in Mean Intensity",
        "(b) Change in Mean Duration",
        "(c) Change in Decadal Frequency"
    ]

    labels = [
        "Δ Whiplash Score",
        "Δ Duration (months)",
        "Δ Frequency (events/decade)"
    ]

    vmax_values = [
        0.25,   # intensity
        10,    # duration
        2.0     # frequency
    ]

    # ------------------------------------------------
    # Marker size for each panel
    # ------------------------------------------------
    marker_sizes = [
        0.5,     # intensity
        0.5,     # duration
        0.5       # frequency
    ]

    plots = []

    for ax, da, title, vmax, marker_size in zip(
        axes,
        datasets,
        titles,
        vmax_values,
        marker_sizes
    ):

        norm = mcolors.TwoSlopeNorm(
            vmin=-vmax,
            vcenter=0,
            vmax=vmax
        )

        # ------------------------------------------------
        # Base raster
        # ------------------------------------------------
        m = da.plot(
            ax=ax,
            cmap="RdBu_r",
            norm=norm,
            add_colorbar=False,
            add_labels=False,
            transform=ccrs.PlateCarree()
        )

        # ------------------------------------------------
        # Scatter overlay
        # ------------------------------------------------
        lon_vals = da["lon"].values
        lat_vals = da["lat"].values

        lon2d, lat2d = np.meshgrid(
            lon_vals,
            lat_vals
        )

        vals = da.values

        mask = (
            np.isfinite(vals)
            & (vals != 0)
        )

        ax.scatter(
            lon2d[mask],
            lat2d[mask],
            c=vals[mask],
            cmap="RdBu_r",
            norm=norm,
            s=marker_size,
            marker="s",
            edgecolors="none",
            transform=ccrs.PlateCarree(),
            zorder=5
        )

        plots.append(m)

        # ------------------------------------------------
        # Map formatting
        # ------------------------------------------------
        ax.set_extent(
            [-180, 180, -90, 90],
            crs=ccrs.PlateCarree()
        )

        ax.coastlines(
            resolution="110m",
            linewidth=0.8
        )

        ax.add_feature(
            cfeature.OCEAN,
            facecolor="lightgrey",
            zorder=0
        )

        ax.gridlines(
            draw_labels=False,
            linewidth=0.4,
            color="black",
            linestyle="--",
            alpha=0.4
        )

        xticks = np.arange(-180, 181, 90)
        yticks = np.arange(-90, 91, 45)

        ax.set_xticks(
            xticks,
            crs=ccrs.PlateCarree()
        )

        ax.set_yticks(
            yticks,
            crs=ccrs.PlateCarree()
        )

        ax.set_xticklabels(
            [
                f"{abs(x)}°E"
                if x >= 0
                else f"{abs(x)}°W"
                for x in xticks
            ],
            fontsize=9,
            fontweight="bold"
        )

        ax.set_yticklabels(
            [
                f"{abs(y)}°N"
                if y >= 0
                else f"{abs(y)}°S"
                for y in yticks
            ],
            fontsize=9,
            fontweight="bold"
        )

        ax.tick_params(
            axis="both",
            direction="in",
            length=5,
            width=1.2
        )

        ax.set_title(
            title,
            fontsize=14,
            fontweight="bold"
        )

        ax.spines["geo"].set_linewidth(1.3)
        ax.spines["geo"].set_edgecolor("black")

    # ------------------------------------------------
    # Individual colorbars
    # ------------------------------------------------
    for ax, m, label in zip(
        axes,
        plots,
        labels
    ):

        cbar = fig.colorbar(
            m,
            ax=ax,
            orientation="horizontal",
            fraction=0.046,
            pad=0.08,
            extend="both"
        )

        cbar.set_label(
            label,
            fontsize=11,
            fontweight="bold"
        )

        cbar.ax.tick_params(
            labelsize=10,
            width=1.1,
            direction="in"
        )

        for tick in cbar.ax.get_xticklabels():
            tick.set_fontweight("bold")

        cbar.outline.set_linewidth(1.2)

    # ------------------------------------------------
    # Save
    # ------------------------------------------------
    if save_path is not None:

        plt.savefig(
            save_path,
            dpi=600,
            bbox_inches="tight"
        )

    plt.show()

In [ ]:
plot_difference_maps(
    diff_int_80_10_da,
    diff_dur_80_10_da,
    diff_freq_80_10_da,
    save_path = r"D:\IITD\Doctoral Thesis\Work and Progress ppts\nature communications\Plots_final\change_2010_2019_1980_1989.tiff"

)

In [ ]:
for name, da in {
    "Intensity": diff_int_80_10_da,
    "Duration": diff_dur_80_10_da,
    "Frequency": diff_freq_80_10_da
}.items():

    vals = da.values
    vals = vals[np.isfinite(vals)]

    print(f"\n{name}")

    for q in [50, 75, 90, 95, 98, 99]:
        print(
            f"{q}th percentile absolute change =",
            np.percentile(np.abs(vals), q)
        )

#### first 20 years v/s last 20 years

In [ ]:
import numpy as np
import pandas as pd

two_decades = {
    "1980-1999": (1980, 1999),
    "2000-2019": (2000, 2019)
}

two_decadal_stats = {
    period: {}
    for period in two_decades
}

for (lat_i, lon_i), events in whiplash_dict_c_e.items():

    for period, (start, end) in two_decades.items():

        freq = 0
        intensities = []
        durations = []

        # ---------------------------------------------
        # Collect all events within this 20-year period
        # ---------------------------------------------
        for e in events:

            year = pd.to_datetime(
                e["tc_time"]
            ).year

            if start <= year <= end:

                freq += 1

                intensities.append(
                    e["score"]
                )

                tc = pd.to_datetime(
                    e["tc_time"]
                )

                td = pd.to_datetime(
                    e["td_time"]
                )

                duration = (
                    (td.year - tc.year) * 12
                    + (td.month - tc.month)
                    + 1
                )

                durations.append(
                    duration
                )

        # ---------------------------------------------
        # Calculate statistics AFTER checking all events
        # ---------------------------------------------
        two_decadal_stats[period][(lat_i, lon_i)] = {

            # Average number of events per decade
            "frequency": freq / 2,

            # Mean intensity of ALL events in the 20-year period
            "intensity": (
                np.mean(intensities)
                if intensities
                else np.nan
            ),

            # Mean duration of ALL events in the 20-year period
            "duration": (
                np.mean(durations)
                if durations
                else np.nan
            )
        }

In [ ]:
def two_decade_to_arrays(
    two_decadal_stats,
    period,
    nlat,
    nlon
):

    intensity = np.full(
        (nlat, nlon),
        np.nan
    )

    duration = np.full(
        (nlat, nlon),
        np.nan
    )

    # Frequency = 0 where no event occurred
    frequency = np.zeros(
        (nlat, nlon),
        dtype=float
    )

    for (i, j), stats in two_decadal_stats[period].items():

        intensity[i, j] = stats["intensity"]
        duration[i, j] = stats["duration"]
        frequency[i, j] = stats["frequency"]

    return intensity, duration, frequency

In [ ]:
i_early20, d_early20, f_early20 = two_decade_to_arrays(two_decadal_stats, "1980-1999", nlat,nlon)
i_late20, d_late20, f_late20 = two_decade_to_arrays(two_decadal_stats,"2000-2019",nlat,nlon)

In [ ]:
# =====================================================
# INTENSITY
# 2000–2019 minus 1980–1999
# =====================================================

diff_int_20yr = (np.nan_to_num(i_late20,nan=0.0)-np.nan_to_num(i_early20,nan=0.0))
# If there were no events in EITHER period,
# leave the grid cell blank
diff_int_20yr[np.isnan(i_early20)&np.isnan(i_late20)] = np.nan

In [ ]:
# =====================================================
# DURATION
# =====================================================

diff_dur_20yr = (np.nan_to_num(d_late20,nan=0.0)-np.nan_to_num(d_early20,nan=0.0))
diff_dur_20yr[np.isnan(d_early20)&np.isnan(d_late20)] = np.nan

In [ ]:
# =====================================================
# FREQUENCY
# =====================================================

diff_freq_20yr = (f_late20-f_early20)
#Blank only if frequency = 0 in BOTH periods
diff_freq_20yr[(f_early20 == 0)&(f_late20 == 0)] = np.nan

In [ ]:
diff_int_20yr_da = make_da(diff_int_20yr,"intensity_change")
diff_dur_20yr_da = make_da(diff_dur_20yr,"duration_change")
diff_freq_20yr_da = make_da(diff_freq_20yr,"frequency_change")

In [ ]:
for name, da in {"Intensity": diff_int_20yr_da,"Duration": diff_dur_20yr_da,"Frequency": diff_freq_20yr_da}.items():
    vals = da.values
    vals = vals[np.isfinite(vals)]

    print(f"\n{name}")
    print("Valid cells:", len(vals))
    print("Min:", np.min(vals))
    print("Max:", np.max(vals))
    print("Mean:", np.mean(vals))
    print("Positive:",np.sum(vals > 0))
    print("Negative:",np.sum(vals < 0))

In [ ]:
def plot_two_decade_difference_maps(
    intensity_da,
    duration_da,
    frequency_da,
    save_path=None
):

    rcParams["font.family"] = "Times New Roman"
    rcParams["font.size"] = 12

    fig = plt.figure(
        figsize=(11.5, 7),
        dpi=600
    )

    # ------------------------------------------------
    # 2 rows × 4 columns
    # top: two maps
    # bottom: one centred map
    # ------------------------------------------------
    gs = fig.add_gridspec(
        2, 4,
        wspace=0.25,
        hspace=0.28
    )

    ax1 = fig.add_subplot(
        gs[0, 0:2],
        projection=ccrs.PlateCarree()
    )

    ax2 = fig.add_subplot(
        gs[0, 2:4],
        projection=ccrs.PlateCarree()
    )

    ax3 = fig.add_subplot(
        gs[1, 1:3],
        projection=ccrs.PlateCarree()
    )

    axes = [ax1, ax2, ax3]

    datasets = [
        intensity_da,
        duration_da,
        frequency_da
    ]

    titles = [
        "(a) Change in Mean Intensity",
        "(b) Change in Mean Duration",
        "(c) Change in Mean Decadal Frequency"
    ]

    labels = [
        "Δ Whiplash Score",
        "Δ Duration (months)",
        "Δ Frequency (events/decade)"
    ]

    vmax_values = [
        0.3,   # intensity
        10.0,    # duration
        2.0     # frequency
    ]

    # ------------------------------------------------
    # Marker size for each panel
    # ------------------------------------------------
    marker_sizes = [
        0.5,     # intensity
        0.5,     # duration
        0.5       # frequency
    ]

    plots = []

    for ax, da, title, vmax, marker_size in zip(
        axes,
        datasets,
        titles,
        vmax_values,
        marker_sizes
    ):

        norm = mcolors.TwoSlopeNorm(
            vmin=-vmax,
            vcenter=0,
            vmax=vmax
        )

        # ------------------------------------------------
        # Base raster
        # ------------------------------------------------
        m = da.plot(
            ax=ax,
            cmap="RdBu_r",
            norm=norm,
            add_colorbar=False,
            add_labels=False,
            transform=ccrs.PlateCarree()
        )

        # ------------------------------------------------
        # Scatter overlay
        # ------------------------------------------------
        lon_vals = da["lon"].values
        lat_vals = da["lat"].values

        lon2d, lat2d = np.meshgrid(
            lon_vals,
            lat_vals
        )

        vals = da.values

        mask = (
            np.isfinite(vals)
            & (vals != 0)
        )

        ax.scatter(
            lon2d[mask],
            lat2d[mask],
            c=vals[mask],
            cmap="RdBu_r",
            norm=norm,
            s=marker_size,
            marker="s",
            edgecolors="none",
            transform=ccrs.PlateCarree(),
            zorder=5
        )

        plots.append(m)

        # ------------------------------------------------
        # Map formatting
        # ------------------------------------------------
        ax.set_extent(
            [-180, 180, -90, 90],
            crs=ccrs.PlateCarree()
        )

        ax.coastlines(
            resolution="110m",
            linewidth=0.8
        )

        ax.add_feature(
            cfeature.OCEAN,
            facecolor="lightgrey",
            zorder=0
        )

        ax.gridlines(
            draw_labels=False,
            linewidth=0.4,
            color="black",
            linestyle="--",
            alpha=0.4
        )

        xticks = np.arange(-180, 181, 90)
        yticks = np.arange(-90, 91, 45)

        ax.set_xticks(
            xticks,
            crs=ccrs.PlateCarree()
        )

        ax.set_yticks(
            yticks,
            crs=ccrs.PlateCarree()
        )

        ax.set_xticklabels(
            [
                f"{abs(x)}°E"
                if x >= 0
                else f"{abs(x)}°W"
                for x in xticks
            ],
            fontsize=9,
            fontweight="bold"
        )

        ax.set_yticklabels(
            [
                f"{abs(y)}°N"
                if y >= 0
                else f"{abs(y)}°S"
                for y in yticks
            ],
            fontsize=9,
            fontweight="bold"
        )

        ax.tick_params(
            axis="both",
            direction="in",
            length=5,
            width=1.2
        )

        ax.set_title(
            title,
            fontsize=14,
            fontweight="bold"
        )

        ax.spines["geo"].set_linewidth(1.3)
        ax.spines["geo"].set_edgecolor("black")

    # ------------------------------------------------
    # Individual colorbars
    # ------------------------------------------------
    for ax, m, label in zip(
        axes,
        plots,
        labels
    ):

        cbar = fig.colorbar(
            m,
            ax=ax,
            orientation="horizontal",
            fraction=0.046,
            pad=0.08,
            extend="both"
        )

        cbar.set_label(
            label,
            fontsize=11,
            fontweight="bold"
        )

        cbar.ax.tick_params(
            labelsize=10,
            width=1.1,
            direction="in"
        )

        for tick in cbar.ax.get_xticklabels():
            tick.set_fontweight("bold")

        cbar.outline.set_linewidth(1.2)

    # ------------------------------------------------
    # Save
    # ------------------------------------------------
    if save_path is not None:

        plt.savefig(
            save_path,
            dpi=600,
            bbox_inches="tight"
        )

    plt.show()

In [ ]:
plot_two_decade_difference_maps(
    diff_int_20yr_da,
    diff_dur_20yr_da,
    diff_freq_20yr_da,
    save_path = r"D:\IITD\Doctoral Thesis\Work and Progress ppts\nature communications\Plots_final\change_2000_2019_1980_1999.tiff"
)

# ECA (Trigger rate) plots

In [ ]:
import pandas as pd

In [ ]:
decade_start1 = '1980-01-01'
decade_end1   = '1989-12-01'
times1 = pd.date_range(decade_start1, decade_end1, freq='MS')  # 120 months
nT1 = len(times1)

In [ ]:
decade_start2 = '1990-01-01'
decade_end2  = '1999-12-01'
times2 = pd.date_range(decade_start2, decade_end2, freq='MS')  # 120 months
nT2 = len(times2)

In [ ]:
decade_start3 = '2000-01-01'
decade_end3  = '2009-12-01'
times3 = pd.date_range(decade_start3, decade_end3, freq='MS')  # 120 months
nT3 = len(times3)

In [ ]:
decade_start4 = '2010-01-01'
decade_end4  = '2019-12-01'
times4 = pd.date_range(decade_start4, decade_end4, freq='MS')  # 120 months
nT4 = len(times4)

In [ ]:
all_times = pd.date_range("1980-01-01", "2019-12-01", freq='MS')  # 480 months

In [ ]:
whiplash_score_map1 = np.full((nT1, nlat, nlon), np.nan)
whiplash_score_map2 = np.full((nT2, nlat, nlon), np.nan)
whiplash_score_map3 = np.full((nT3, nlat, nlon), np.nan)
whiplash_score_map4 = np.full((nT4, nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():
    for ev in ev_list:
        # Get the actual event time range
        ev_start_time = all_times[ev["tc_index"]]
        ev_end_time   = all_times[ev["td_index"]]
        
        # Check if event overlaps with the decade
        if ev_end_time < pd.Timestamp(decade_start1) or ev_start_time > pd.Timestamp(decade_end1):
            continue  # skip events outside this decade

        # Clip event to the decade
        ev_start_time = max(ev_start_time, pd.Timestamp(decade_start1))
        ev_end_time   = min(ev_end_time, pd.Timestamp(decade_end1))
        
        # Convert back to indices relative to decade
        t_start = times1.get_loc(ev_start_time)
        t_end   = times1.get_loc(ev_end_time) + 1

        for t_idx in range(t_start, t_end):
            if np.isnan(whiplash_score_map1[t_idx, i, j]):
                whiplash_score_map1[t_idx, i, j] = ev["score"]
            else:
                whiplash_score_map1[t_idx, i, j] = max(
                    whiplash_score_map1[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():
    for ev in ev_list:
        # Get the actual event time range
        ev_start_time = all_times[ev["tc_index"]]
        ev_end_time   = all_times[ev["td_index"]]
        
        # Check if event overlaps with the decade
        if ev_end_time < pd.Timestamp(decade_start2) or ev_start_time > pd.Timestamp(decade_end2):
            continue  # skip events outside this decade

        # Clip event to the decade
        ev_start_time = max(ev_start_time, pd.Timestamp(decade_start2))
        ev_end_time   = min(ev_end_time, pd.Timestamp(decade_end2))
        
        # Convert back to indices relative to decade
        t_start = times2.get_loc(ev_start_time)
        t_end   = times2.get_loc(ev_end_time) + 1

        for t_idx in range(t_start, t_end):
            if np.isnan(whiplash_score_map2[t_idx, i, j]):
                whiplash_score_map2[t_idx, i, j] = ev["score"]
            else:
                whiplash_score_map2[t_idx, i, j] = max(
                    whiplash_score_map2[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():
    for ev in ev_list:
        # Get the actual event time range
        ev_start_time = all_times[ev["tc_index"]]
        ev_end_time   = all_times[ev["td_index"]]
        
        # Check if event overlaps with the decade
        if ev_end_time < pd.Timestamp(decade_start3) or ev_start_time > pd.Timestamp(decade_end3):
            continue  # skip events outside this decade

        # Clip event to the decade
        ev_start_time = max(ev_start_time, pd.Timestamp(decade_start3))
        ev_end_time   = min(ev_end_time, pd.Timestamp(decade_end3))
        
        # Convert back to indices relative to decade
        t_start = times3.get_loc(ev_start_time)
        t_end   = times3.get_loc(ev_end_time) + 1

        for t_idx in range(t_start, t_end):
            if np.isnan(whiplash_score_map3[t_idx, i, j]):
                whiplash_score_map3[t_idx, i, j] = ev["score"]
            else:
                whiplash_score_map3[t_idx, i, j] = max(
                    whiplash_score_map3[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():
    for ev in ev_list:
        # Get the actual event time range
        ev_start_time = all_times[ev["tc_index"]]
        ev_end_time   = all_times[ev["td_index"]]
        
        # Check if event overlaps with the decade
        if ev_end_time < pd.Timestamp(decade_start4) or ev_start_time > pd.Timestamp(decade_end4):
            continue  # skip events outside this decade

        # Clip event to the decade
        ev_start_time = max(ev_start_time, pd.Timestamp(decade_start4))
        ev_end_time   = min(ev_end_time, pd.Timestamp(decade_end4))
        
        # Convert back to indices relative to decade
        t_start = times4.get_loc(ev_start_time)
        t_end   = times4.get_loc(ev_end_time) + 1

        for t_idx in range(t_start, t_end):
            if np.isnan(whiplash_score_map4[t_idx, i, j]):
                whiplash_score_map4[t_idx, i, j] = ev["score"]
            else:
                whiplash_score_map4[t_idx, i, j] = max(
                    whiplash_score_map4[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
whiplash_score_da1 = xr.DataArray(
    data=whiplash_score_map1,          # numpy array with all values filled
    coords={
        'time': times1,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
whiplash_score_da2 = xr.DataArray(
    data=whiplash_score_map2,          # numpy array with all values filled
    coords={
        'time': times2,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
whiplash_score_da3 = xr.DataArray(
    data=whiplash_score_map3,          # numpy array with all values filled
    coords={
        'time': times3,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
whiplash_score_da4 = xr.DataArray(
    data=whiplash_score_map4,          # numpy array with all values filled
    coords={
        'time': times4,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
water_stress_ds =xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\Water Stress\water_stress_1980_2019.nc")

In [ ]:
water_stress_da = water_stress_ds["__xarray_dataarray_variable__"]

In [ ]:
# Select baseline period
baseline_stress = water_stress_da.sel(time=slice("1995-01-01", "2014-12-31"))

In [ ]:
# Compute 90th percentile per grid point
ST20 = baseline_stress.quantile(0.20, dim="time")

# P90 dims: (lat, lon)
print(ST20.dims)

In [ ]:
ws_decade1 = water_stress_da.sel(time=slice('1980-01-01', '1989-12-31'))

In [ ]:
ws_decade2 = water_stress_da.sel(time=slice('1990-01-01', '1999-12-31'))

In [ ]:
ws_decade3 = water_stress_da.sel(time=slice('2000-01-01', '2009-12-31'))

In [ ]:
ws_decade4 = water_stress_da.sel(time=slice('2010-01-01', '2019-12-31'))

In [ ]:
def ECA_trigger_grid(X, Y, deltaT=5, tau=0):
    """
    X : 1D numpy array (binary whiplash indicator, 1 during whiplash sequence)
    Y : 1D numpy array (binary water stress indicator, 1 during stress)
    deltaT : maximum allowed delay after whiplash sequence ends (months)
    tau : minimum lag after whiplash sequence ends (default 0)

    Returns:
    r_trig : trigger coincidence rate
    """

    import numpy as np

    # indices where whiplash exists
    tX = np.where(X == 1)[0]
    tY = np.where(Y == 1)[0]

    NX = len(tX)
    NY = len(tY)

    if NX == 0 or NY == 0:
        return np.nan

    # Identify contiguous whiplash sequences
    sequences = []
    start = tX[0]

    for i in range(1, NX):
        if tX[i] != tX[i-1] + 1:
            sequences.append((start, tX[i-1]))
            start = tX[i]

    sequences.append((start, tX[-1]))

    # Check stress AFTER sequence end
    triggered = 0
    total_sequences = len(sequences)

    for seq in sequences:

        tc, td = seq   # td = end of whiplash sequence

        # define allowed window AFTER sequence
        start_window = td + tau
        end_window   = td + tau + deltaT

        # ensure within bounds
        start_window = max(start_window, 0)
        end_window   = min(end_window, len(Y)-1)

        if start_window <= end_window:

            if np.any(Y[start_window:end_window+1] == 1):
                triggered += 1

    r_trig = triggered / total_sequences

    return r_trig

In [ ]:
whiplash_month1 = (whiplash_score_da1 > 0).astype(int)
stress_month1 = (ws_decade1 < ST20).astype(int)

In [ ]:
whiplash_month2 = (whiplash_score_da2 > 0).astype(int)
stress_month2 = (ws_decade2 < ST20).astype(int)

In [ ]:
whiplash_month3 = (whiplash_score_da3 > 0).astype(int)
stress_month3 = (ws_decade3 < ST20).astype(int)

In [ ]:
whiplash_month4 = (whiplash_score_da4 > 0).astype(int)
stress_month4 = (ws_decade4 < ST20).astype(int)

In [ ]:
lat_size = whiplash_month1.latitude.size
lon_size = whiplash_month1.longitude.size

trigger_rate_decade1 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month1[:, i, j].values
        Y = stress_month1[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_decade1[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
lat_size = whiplash_month2.latitude.size
lon_size = whiplash_month2.longitude.size

trigger_rate_decade2 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month2[:, i, j].values
        Y = stress_month2[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_decade2[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
lat_size = whiplash_month3.latitude.size
lon_size = whiplash_month3.longitude.size

trigger_rate_decade3 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month3[:, i, j].values
        Y = stress_month3[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_decade3[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
lat_size = whiplash_month4.latitude.size
lon_size = whiplash_month4.longitude.size

trigger_rate_decade4 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month4[:, i, j].values
        Y = stress_month4[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_decade4[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
# convert to xarray DataArray
trigger_rate_da_1 = xr.DataArray(
    data=trigger_rate_decade1,
    coords={'latitude': whiplash_month1.latitude, 'longitude': whiplash_month1.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
# convert to xarray DataArray
trigger_rate_da_2 = xr.DataArray(
    data=trigger_rate_decade2,
    coords={'latitude': whiplash_month2.latitude, 'longitude': whiplash_month2.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
# convert to xarray DataArray
trigger_rate_da_3 = xr.DataArray(
    data=trigger_rate_decade3,
    coords={'latitude': whiplash_month3.latitude, 'longitude': whiplash_month3.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
# convert to xarray DataArray
trigger_rate_da_4 = xr.DataArray(
    data=trigger_rate_decade4,
    coords={'latitude': whiplash_month4.latitude, 'longitude': whiplash_month4.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
import numpy as np

def trigger_rate_significance(X, Y, tau=0, delT=5, n_iter=1000, seed=None):
    """
    X, Y : 1D binary arrays
    tau : lag
    delT : window length
    n_iter : number of Monte Carlo simulations
    Returns: observed rate, p-value
    """
    np.random.seed(seed)
    
    # observed trigger rate
    r_obs = ECA_trigger_grid(X, Y, deltaT=5, tau=0)
    
    # number of events
    NX = X.sum()
    NY = Y.sum()
    
    if NX == 0 or NY == 0:
        return r_obs, np.nan
    
    r_null = []
    T = len(X)
    
    for _ in range(n_iter):
        # randomly place X and Y events in time
        X_rand = np.zeros(T, dtype=int)
        Y_rand = np.zeros(T, dtype=int)
        
        X_idx = np.random.choice(T, NX, replace=False)
        Y_idx = np.random.choice(T, NY, replace=False)
        
        X_rand[X_idx] = 1
        Y_rand[Y_idx] = 1
        
        r_null.append(ECA_trigger_grid(X_rand, Y_rand, tau))
    
    r_null = np.array(r_null)
    
    # p-value: fraction of random trigger rates >= observed
    p_value = (r_null >= r_obs).sum() / n_iter
    
    return r_obs, p_value

In [ ]:
p_value_map_1 = np.full((lat_size, lon_size), np.nan)
trigger_rate_map_1 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month1[:, i, j].values
        Y = stress_month1[:, i, j].values
        
        r_obs_1, p_val_1 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_rate_map_1[i, j] = r_obs_1
        p_value_map_1[i, j] = p_val_1

In [ ]:
p_value_map_2 = np.full((lat_size, lon_size), np.nan)
trigger_rate_map_2 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month2[:, i, j].values
        Y = stress_month2[:, i, j].values
        
        r_obs_2, p_val_2 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_rate_map_2[i, j] = r_obs_2
        p_value_map_2[i, j] = p_val_2

In [ ]:
p_value_map_3 = np.full((lat_size, lon_size), np.nan)
trigger_rate_map_3 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month3[:, i, j].values
        Y = stress_month3[:, i, j].values
        
        r_obs_3, p_val_3 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_rate_map_3[i, j] = r_obs_3
        p_value_map_3[i, j] = p_val_3

In [ ]:
p_value_map_4 = np.full((lat_size, lon_size), np.nan)
trigger_rate_map_4 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month4[:, i, j].values
        Y = stress_month4[:, i, j].values
        
        r_obs_4, p_val_4 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_rate_map_4[i, j] = r_obs_4
        p_value_map_4[i, j] = p_val_4

In [ ]:
# convert to xarray DataArray
p_value_da1 = xr.DataArray(
    data=p_value_map_1,
    coords={'latitude': whiplash_month1.latitude, 'longitude': whiplash_month1.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

In [ ]:
# convert to xarray DataArray
p_value_da2 = xr.DataArray(
    data=p_value_map_2,
    coords={'latitude': whiplash_month2.latitude, 'longitude': whiplash_month2.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

In [ ]:
# convert to xarray DataArray
p_value_da3 = xr.DataArray(
    data=p_value_map_3,
    coords={'latitude': whiplash_month3.latitude, 'longitude': whiplash_month3.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

In [ ]:
# convert to xarray DataArray
p_value_da4 = xr.DataArray(
    data=p_value_map_4,
    coords={'latitude': whiplash_month4.latitude, 'longitude': whiplash_month4.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

In [ ]:
# Keep only values where p < 0.05 (significant)
trigger_rate_significant_da_1= trigger_rate_da_1.where(p_value_da1 < 0.05)
trigger_rate_significant_da_2= trigger_rate_da_2.where(p_value_da2 < 0.05)
trigger_rate_significant_da_3= trigger_rate_da_3.where(p_value_da3 < 0.05)
trigger_rate_significant_da_4= trigger_rate_da_4.where(p_value_da4 < 0.05)

In [ ]:
# Keep only values where p < 0.05 (significant)
trigger_rate_significant_da_1.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_1980_1989.nc")

In [ ]:
trigger_rate_significant_da_2.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_1990_1999.nc")
trigger_rate_significant_da_3.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_2000_2009.nc")
trigger_rate_significant_da_4.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_2010_2019.nc")

In [ ]:
trigger_rate_significant_da_1

In [ ]:
# Keep only values where p < 0.05 (significant)
trigger_rate_significant_1 = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_1980_1989.nc")
trigger_rate_significant_2 = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_1990_1999.nc")
trigger_rate_significant_3 = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_2000_2009.nc")
trigger_rate_significant_4 = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_2010_2019.nc")

In [ ]:
trigger_rate_significant_da_1 = trigger_rate_significant_1['trigger_coincidence_rate']
trigger_rate_significant_da_2 = trigger_rate_significant_2['trigger_coincidence_rate']
trigger_rate_significant_da_3 = trigger_rate_significant_3['trigger_coincidence_rate']
trigger_rate_significant_da_4 = trigger_rate_significant_4['trigger_coincidence_rate']

In [ ]:
trigger_rate_significant_1

In [ ]:
count = (trigger_rate_significant_1["trigger_coincidence_rate"] > 0).sum().item()

print(count)

In [ ]:
count1 = (trigger_rate_significant_4["trigger_coincidence_rate"] > 0).sum().item()

print(count1)

In [ ]:
import matplotlib as mpl

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# --- Settings ---
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.linewidth"] = 1.8  # global border thickness

decades = ["1980-1989","1990-1999","2000-2009","2010-2019"]
decadal_tcr = [
    trigger_rate_significant_da_1,
    trigger_rate_significant_da_2,
    trigger_rate_significant_da_3,
    trigger_rate_significant_da_4
]

lat_values = trigger_rate_significant_da_1.latitude.values
lon_values = trigger_rate_significant_da_1.longitude.values

# Colormap
cmap = "coolwarm"
norm = mcolors.Normalize(vmin=0, vmax=1)

# Figure layout
fig = plt.figure(figsize=(24,3.5), dpi=300)
gs = gridspec.GridSpec(1, 5, width_ratios=[1,1,1,1,0.05], wspace=0.15)

axes = []
for i in range(4):
    ax = fig.add_subplot(gs[0,i], projection=ccrs.PlateCarree())
    axes.append(ax)

# Meshgrid
lon2d, lat2d = np.meshgrid(lon_values, lat_values)

# Ticks
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

# ---- Plot each panel ----
for i, ax in enumerate(axes):
    
    da = decadal_tcr[i]
    mask = ~np.isnan(da.values)

    sc = ax.scatter(
        lon2d[mask],
        lat2d[mask],
        c=da.values[mask],
        s=2,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        edgecolors="none"
    )

    # Map styling
    ax.set_extent([-180, 180, -90, 90])
    ax.coastlines(resolution="110m", linewidth=0.6)
    ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")

    # Gridlines (no labels)
    gl = ax.gridlines(draw_labels=False, linewidth=0.4, linestyle="--", alpha=0.5)

    # ---- TICKS WITH LABELS ----
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)

    ax.set_xticklabels(
        [f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks],
        fontsize=11, fontweight="bold"
    )
    ax.set_yticklabels(
        [f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks],
        fontsize=11, fontweight="bold"
    )

    # ---- THICK BORDERS ----
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2.0)

    # Titles
    ax.set_title(decades[i], fontweight="bold", fontsize=13, pad=6)

# ---- COLORBAR ----
# ---- COLORBAR ----
cax = fig.add_subplot(gs[0,4])

# Get original position
pos = cax.get_position()

# Shrink height (e.g., 70%) and center it
cax.set_position([
    pos.x0,                 # same x
    pos.y0 + pos.height*0.15,  # move up a bit
    pos.width,              # same width
    pos.height*0.7          # reduce height
])

cbar = fig.colorbar(sc, cax=cax, orientation="vertical")

cbar.set_label("Trigger Coincidence Rate",
               fontsize=12, fontweight="bold")

cbar.ax.tick_params(labelsize=11, width=1.2)
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

cbar.outline.set_linewidth(1.2)
cbar.outline.set_linewidth(1.5)

#Save
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\Decadal_trigger_coincidence.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# --- Settings ---
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.linewidth"] = 1.8  # global border thickness

decades = ["1980-1989","2010-2019"]
decadal_tcr = [
    trigger_rate_significant_da_1,
    trigger_rate_significant_da_4
]

lat_values = trigger_rate_significant_da_1.latitude.values
lon_values = trigger_rate_significant_da_1.longitude.values

# Colormap
cmap = "coolwarm"
norm = mcolors.Normalize(vmin=0, vmax=1)

# Figure layout
fig = plt.figure(figsize=(12,4), dpi=300)
gs = gridspec.GridSpec(
    1, 3,
    width_ratios=[1, 1, 0.05],
    wspace = 0.22
)

axes = []
for i in range(2):
    ax = fig.add_subplot(gs[0, i], projection=ccrs.PlateCarree())
    axes.append(ax)

# Meshgrid
lon2d, lat2d = np.meshgrid(lon_values, lat_values)

# Ticks
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

# ---- Plot each panel ----
for i, ax in enumerate(axes):
    
    da = decadal_tcr[i]
    mask = ~np.isnan(da.values)

    sc = ax.scatter(
        lon2d[mask],
        lat2d[mask],
        c=da.values[mask],
        s=2,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        edgecolors="none"
    )

    # Map styling
    ax.set_extent([-180, 180, -90, 90])
    ax.coastlines(resolution="110m", linewidth=0.6)
    ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")

    # Gridlines (no labels)
    gl = ax.gridlines(draw_labels=False, linewidth=0.4, linestyle="--", alpha=0.5)

    # ---- TICKS WITH LABELS ----
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)

    ax.set_xticklabels(
        [f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks],
        fontsize=11, fontweight="bold"
    )
    ax.set_yticklabels(
        [f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks],
        fontsize=11, fontweight="bold"
    )

    # ---- THICK BORDERS ----
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.8)

    # Titles
    ax.set_title(
    f"({chr(97+i)}) {decades[i]}",
    fontweight="bold",
    fontsize=14,
    pad=6
)

# ---- COLORBAR ----
# ---- COLORBAR ----
cax = fig.add_subplot(gs[0,2])

# Get original position
pos = cax.get_position()

# Shrink height (e.g., 70%) and center it
cax.set_position([
    pos.x0,                 # same x
    pos.y0 + pos.height*0.15,  # move up a bit
    pos.width,              # same width
    pos.height*0.7          # reduce height
])

cbar = fig.colorbar(sc, cax=cax, orientation="vertical")

cbar.set_label("Trigger Coincidence Rate",
               fontsize=12, fontweight="bold")

cbar.ax.tick_params(labelsize=11, width=1.2)
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

cbar.outline.set_linewidth(1.2)
cbar.outline.set_linewidth(1.5)

#Save
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\First_Last_Decadal_trigger_coincidence.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# --- Settings ---
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.linewidth"] = 1.8  # global border thickness

decades = ["1980-1989","2010-2019"]
decadal_tcr = [
    trigger_rate_significant_da_1,
    trigger_rate_significant_da_4
]

lat_values = trigger_rate_significant_da_1.latitude.values
lon_values = trigger_rate_significant_da_1.longitude.values

# Colormap
cmap = "coolwarm"
norm = mcolors.Normalize(vmin=0, vmax=1)

# Figure layout
fig = plt.figure(figsize=(6,6), dpi=300)
gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[1, 0.05],
    hspace = 0.22,
    wspace = 0.12,
)

# Map axes
ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree())

axes = [ax1, ax2]

# Meshgrid
lon2d, lat2d = np.meshgrid(lon_values, lat_values)

# Ticks
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

# ---- Plot each panel ----
for i, ax in enumerate(axes):
    
    da = decadal_tcr[i]
    mask = ~np.isnan(da.values)

    sc = ax.scatter(
        lon2d[mask],
        lat2d[mask],
        c=da.values[mask],
        s=2,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        edgecolors="none"
    )

    # Map styling
    ax.set_extent([-180, 180, -90, 90])
    ax.coastlines(resolution="110m", linewidth=0.6)
    ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")

    # Gridlines (no labels)
    gl = ax.gridlines(draw_labels=False, linewidth=0.4, linestyle="--", alpha=0.5)

    # ---- TICKS WITH LABELS ----
    ax.set_xticks(xticks)
    ax.set_yticks(yticks)

    ax.set_xticklabels(
        [f"{abs(x)}°E" if x>=0 else f"{abs(x)}°W" for x in xticks],
        fontsize=11, fontweight="bold"
    )
    ax.set_yticklabels(
        [f"{abs(y)}°N" if y>=0 else f"{abs(y)}°S" for y in yticks],
        fontsize=11, fontweight="bold"
    )

    # ---- THICK BORDERS ----
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.8)

    # Titles
    ax.set_title(
    f"({chr(97+i)}) {decades[i]}",
    fontweight="bold",
    fontsize=14,
    pad=6
)

# ---- COLORBAR ----
# ---- COLORBAR ----
cax = fig.add_subplot(gs[:,1])

# Get original position
pos = cax.get_position()

# Shrink height (e.g., 70%) and center it
cax.set_position([
    pos.x0,                 # same x
    pos.y0 + pos.height*0.12,  # move up a bit
    pos.width,              # same width
    pos.height*0.76         # reduce height
])

cbar = fig.colorbar(sc, cax=cax, orientation="vertical")

cbar.set_label("Trigger Coincidence Rate",
               fontsize=12, fontweight="bold")

cbar.ax.tick_params(labelsize=11, width=1.2)
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

cbar.outline.set_linewidth(1.2)
cbar.outline.set_linewidth(1.5)

#Save
plt.savefig(
    r"D:\IITD\Doctoral Thesis\Work and Progress ppts\nature communications\Plots_final\First_Last_Decadal_tr_one_below_other_compressed.tiff",
    dpi=50,
    bbox_inches="tight"
)

plt.show()

# Adaptation Relevant Hotspots (2010-2019)

In [ ]:
decadal_stats

In [ ]:
lat_size = len(trigger_rate_significant_da_1.latitude)
lon_size = len(trigger_rate_significant_da_1.longitude)

In [ ]:
# Function to convert decadal_stats to 2D grid
def decadal_dict_to_grid(decadal_dict, metric, lat_size, lon_size):
    grid = np.full((lat_size, lon_size), np.nan, dtype=float)
    for (i, j), vals in decadal_dict.items():
        grid[i, j] = vals[metric]
    return grid

In [ ]:
# Choose decade
decade1 = "1980-1989"
stats1 = decadal_stats[decade1]

# Convert to 2D grids
freq_grid1 = decadal_dict_to_grid(stats1, "frequency", lat_size, lon_size)
intensity_grid1 = decadal_dict_to_grid(stats1, "intensity", lat_size, lon_size)
duration_grid1 = decadal_dict_to_grid(stats1, "duration", lat_size, lon_size)

In [ ]:
freq_grid1

In [ ]:
import numpy as np

# Example: your 2D grids
# freq_grid, intensity_grid, duration_grid

# Check if all values are zero or NaN
def all_zero_or_nan(grid):
    return np.all((grid == 0) | np.isnan(grid))

print("Frequency grid all zero or NaN:", all_zero_or_nan(freq_grid2))
print("Intensity grid all zero or NaN:", all_zero_or_nan(intensity_grid2))
print("Duration grid all zero or NaN:", all_zero_or_nan(duration_grid2))

In [ ]:
# Choose decade
decade2 = "1990-1999"
stats2 = decadal_stats[decade2]

# Convert to 2D grids
freq_grid2 = decadal_dict_to_grid(stats2, "frequency", lat_size, lon_size)
intensity_grid2 = decadal_dict_to_grid(stats2, "intensity", lat_size, lon_size)
duration_grid2 = decadal_dict_to_grid(stats2, "duration", lat_size, lon_size)

In [ ]:
# Choose decade
decade3 = "2000-2009"
stats3 = decadal_stats[decade3]

# Convert to 2D grids
freq_grid3 = decadal_dict_to_grid(stats3, "frequency", lat_size, lon_size)
intensity_grid3 = decadal_dict_to_grid(stats3, "intensity", lat_size, lon_size)
duration_grid3 = decadal_dict_to_grid(stats3, "duration", lat_size, lon_size)

In [ ]:
# Choose decade
decade4 = "2010-2019"
stats4 = decadal_stats[decade4]

# Convert to 2D grids
freq_grid4 = decadal_dict_to_grid(stats4, "frequency", lat_size, lon_size)
intensity_grid4 = decadal_dict_to_grid(stats4, "intensity", lat_size, lon_size)
duration_grid4 = decadal_dict_to_grid(stats4, "duration", lat_size, lon_size)

In [ ]:
# --- Normalize each metric ---
def normalize_grid(grid):
    mask = ~np.isnan(grid)
    norm_grid = np.full_like(grid, np.nan, dtype=float)
    if np.any(mask):
        g_min, g_max = np.nanmin(grid[mask]), np.nanmax(grid[mask])
        if g_max - g_min > 0:
            norm_grid[mask] = (grid[mask] - g_min) / (g_max - g_min)
        else:
            norm_grid[mask] = 0.0
    return norm_grid

In [ ]:
freq_norm1 = normalize_grid(freq_grid1)
intensity_norm1 = normalize_grid(intensity_grid1)
duration_norm1 = normalize_grid(duration_grid1)

In [ ]:
print("Frequency grid all zero or NaN:", all_zero_or_nan(freq_norm1))
print("Intensity grid all zero or NaN:", all_zero_or_nan(intensity_norm1))
print("Duration grid all zero or NaN:", all_zero_or_nan(duration_norm1))

In [ ]:
freq_norm2 = normalize_grid(freq_grid2)
intensity_norm2 = normalize_grid(intensity_grid2)
duration_norm2 = normalize_grid(duration_grid2)

In [ ]:
freq_norm3 = normalize_grid(freq_grid3)
intensity_norm3 = normalize_grid(intensity_grid3)
duration_norm3 = normalize_grid(duration_grid3)

In [ ]:
freq_norm4 = normalize_grid(freq_grid4)
intensity_norm4 = normalize_grid(intensity_grid4)
duration_norm4 = normalize_grid(duration_grid4)

In [ ]:
# --- Compute composite IDF ---
composite_idf1 = np.nanmean(np.array([freq_norm1, intensity_norm1, duration_norm1]), axis=0)

In [ ]:
   # ignores NaNs
np.nanmin(composite_idf1)   # ignores NaNs

In [ ]:
np.nanmax(composite_idf1)

In [ ]:
print("composite grid all zero or NaN:", all_zero_or_nan(composite_idf1))

In [ ]:
# --- Compute composite IDF ---
composite_idf2 = np.nanmean(np.array([freq_norm2, intensity_norm2, duration_norm2]), axis=0)

In [ ]:
np.nanmax(composite_idf2)

In [ ]:
# --- Compute composite IDF ---
composite_idf3 = np.nanmean(np.array([freq_norm3, intensity_norm3, duration_norm3]), axis=0)

In [ ]:
np.nanmax(composite_idf3)

In [ ]:
# --- Compute composite IDF ---
composite_idf4 = np.nanmean(np.array([freq_norm4, intensity_norm4, duration_norm4]), axis=0)

In [ ]:
np.nanmax(composite_idf4)

In [ ]:
idf_threshold = np.nanpercentile(composite_idf4, 72)

In [ ]:
idf_threshold

In [ ]:
trigger_rate_significant_da_4.quantile(0.2)

In [ ]:
tcr_threshold = np.nanpercentile(trigger_rate_da.values, 75)

hotspot_mask = (composite_idf >= idf_threshold) & (trigger_rate_da.values >= tcr_threshold)

In [ ]:
# --- Mask hotspots where composite IDF > 0 AND trigger coincidence rate > 0 ---
hotspot_mask1 = (composite_idf1 > 0.1) & (trigger_rate_significant_da_1.values > 0)

In [ ]:
hotspot_mask2 = (composite_idf2 > 0) & (trigger_rate_significant_da_2.values > 0)
hotspot_mask3 = (composite_idf3 > 0) & (trigger_rate_significant_da_3.values > 0)

In [ ]:
hotspot_mask4 = (composite_idf4 > 0.2) & (trigger_rate_significant_da_4.values > 0.5)

In [ ]:
# --- Convert to Xarray for easy plotting ---
composite_da4 = xr.DataArray(
    composite_idf4,
    coords={"latitude": trigger_rate_significant_da_1.latitude, "longitude": trigger_rate_significant_da_1.longitude},
    dims=["latitude","longitude"],
    name="composite_idf"
)

In [ ]:
hotspot_mask4 = (composite_da4 > 0.1) & (trigger_rate_significant_da_4.values > 0.5)

In [ ]:
hotspot_da4 = xr.DataArray(
    hotspot_mask4.astype(int),
    coords={"latitude": trigger_rate_significant_da_1.latitude, "longitude": trigger_rate_significant_da_1.longitude},
    dims=["latitude","longitude"],
    name="hotspot_mask"
)


In [ ]:
hotspot_da4.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\Hotspot_mask_2010_2019.nc")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D

# --- Settings ---
mpl.rcParams["font.family"] = "Times New Roman"
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.linewidth"] = 1.8

fig, ax = plt.subplots(
    1, 1,
    subplot_kw={'projection': ccrs.PlateCarree()},
    figsize=(10, 7)
)

# Map setup
ax.set_extent([-180, 180, -90, 90])
ax.coastlines(resolution='10m', linewidth=1)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# Extract lat/lon
lat_vals = hotspot_da4['latitude'].values
lon_vals = hotspot_da4['longitude'].values
lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

# Mask
mask = hotspot_da4.values == True
lon_pts = lon2d[mask]
lat_pts = lat2d[mask]

# Scatter plot
ax.scatter(
    lon_pts,
    lat_pts,
    color='red',
    s=12,
    marker='o',
    edgecolors='black',
    linewidths=0.3,
    transform=ccrs.PlateCarree(),
    zorder=5
)

# ---- GRIDLINES ----
gl = ax.gridlines(draw_labels=False, linewidth=0.4,
                  linestyle="--", alpha=0.5)

# ---- TICKS ----
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax.set_xticks(xticks)
ax.set_yticks(yticks)

ax.set_xticklabels(
    [f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
    fontsize=12, fontweight="bold"
)
ax.set_yticklabels(
    [f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
    fontsize=12, fontweight="bold"
)

# ---- THICK BORDERS ----
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.8)

# Title
ax.set_title(
    'Adaptation relevant Hotspots (2010–2019)',
    fontsize=14,
    weight='bold'
)

# ---- LEGEND ----
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           label='Hotspots',
           markerfacecolor='red',
           markeredgecolor='black',
           markersize=8)
]

ax.legend(
    handles=legend_elements,
    loc='lower left',
    frameon=True,
    prop={'size': 14, 'weight': 'bold'}
)

# Count
print("Number of hotspot grid cells:", np.sum(mask))

# Save
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\Adaptation_relevant_hotspots_2010_2019.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Adaptation relevant hotspots (1980-2019)

In [ ]:
times = pd.date_range("1980-01-01", "2019-12-01", freq="MS")  # 480 months
nT = len(times)

In [ ]:
whiplash_score_map = np.full((nT, nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():

    for ev in ev_list:

        # full whiplash sequence
        t_start = ev["tc_index"]
        t_end   = ev["td_index"] + 1   # +1 because Python range excludes last index

        for t_idx in range(t_start, t_end):

            if np.isnan(whiplash_score_map[t_idx, i, j]):
                whiplash_score_map[t_idx, i, j] = ev["score"]
            else:
                whiplash_score_map[t_idx, i, j] = max(
                    whiplash_score_map[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
whiplash_score_da = xr.DataArray(
    data=whiplash_score_map,          # numpy array with all values filled
    coords={
        'time': times,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
whiplash_month = (whiplash_score_da > 0).astype(int)

In [ ]:
stress_month = (water_stress_da < ST20).astype(int)

In [ ]:
lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

trigger_rate_map_3 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_map_3[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
# convert to xarray DataArray
trigger_rate_da_3 = xr.DataArray(
    data=trigger_rate_map_3,
    coords={'latitude': whiplash_month.latitude, 'longitude': whiplash_month.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

p_value_map_3 = np.full((lat_size, lon_size), np.nan)
trigger_rate_map_3 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        r_obs_3, p_val_3 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_rate_map_3[i, j] = r_obs_3
        p_value_map_3[i, j] = p_val_3

In [ ]:
# convert to xarray DataArray
p_value_da_3 = xr.DataArray(
    data=p_value_map_3,
    coords={'latitude': whiplash_month.latitude, 'longitude': whiplash_month.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

In [ ]:
# Keep only values where p < 0.05 (significant)
trigger_rate_significant_da = trigger_rate_da_3.where(p_value_da_3 < 0.05)

In [ ]:
import numpy as np

freq_grid = np.zeros((nlat, nlon))
intensity_grid = np.full((nlat, nlon), np.nan)
duration_grid = np.full((nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():

    if len(ev_list) == 0:
        continue

    # -----------------
    # Frequency
    # -----------------
    freq = len(ev_list)
    avg_freq = freq/40
    freq_grid[i, j] = avg_freq

    # -----------------
    # Intensity
    # -----------------
    intensities = [ev["score"] for ev in ev_list]
    intensity_grid[i, j] = np.mean(intensities)

    # -----------------
    # Duration
    # -----------------
    durations = [
        ev["td_index"] - ev["tc_index"] + 1
        for ev in ev_list
    ]
    duration_grid[i, j] = np.mean(durations)

In [ ]:
import xarray as xr

freq_da = xr.DataArray(
    freq_grid,
    coords={'latitude': lat_values, 'longitude': lon_values},
    dims=('latitude', 'longitude'),
    name='frequency'
)

intensity_da = xr.DataArray(
    intensity_grid,
    coords={'latitude': lat_values, 'longitude': lon_values},
    dims=('latitude', 'longitude'),
    name='mean_intensity'
)

duration_da = xr.DataArray(
    duration_grid,
    coords={'latitude': lat_values, 'longitude': lon_values},
    dims=('latitude', 'longitude'),
    name='mean_duration'
)

In [ ]:
# --- Normalize each metric ---
def normalize_grid(grid):
    mask = ~np.isnan(grid)
    norm_grid = np.full_like(grid, np.nan, dtype=float)
    if np.any(mask):
        g_min, g_max = np.nanmin(grid[mask]), np.nanmax(grid[mask])
        if g_max - g_min > 0:
            norm_grid[mask] = (grid[mask] - g_min) / (g_max - g_min)
        else:
            norm_grid[mask] = 0.0
    return norm_grid

In [ ]:
freq_norm = normalize_grid(freq_grid)
intensity_norm = normalize_grid(intensity_grid)
duration_norm = normalize_grid(duration_grid)

In [ ]:
# --- Compute composite IDF ---
composite_idf = np.nanmean(np.array([freq_norm, intensity_norm, duration_norm]), axis=0)

In [ ]:
composite_idf.max()

In [ ]:
composite_da = xr.DataArray(
    composite_idf,
    coords={"latitude": trigger_rate_significant_da.latitude, "longitude": trigger_rate_significant_da.longitude},
    dims=["latitude","longitude"],
    name="composite_idf"
)

In [ ]:
hotspot_mask = (composite_da > 0.15) & (trigger_rate_significant_da.values > 0.5)

In [ ]:
idf_threshold = np.nanpercentile(composite_idf, 9)
idf_threshold 

In [ ]:
hotspot_da = xr.DataArray(
    hotspot_mask.astype(int),
    coords={"latitude": trigger_rate_significant_da.latitude, "longitude": trigger_rate_significant_da.longitude},
    dims=["latitude","longitude"],
    name="hotspot_mask"
)

In [ ]:
hotspot_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\Hotspot_mask_1980_2019.nc")

In [ ]:
import xarray as xr
hotspot = xr.open_dataset(r"\\10.230.28.136\cez248024\Objective 1\Whiplash results\Cause-Effect\Hotspot_mask_1980_2019.nc")

In [ ]:
hotspot_da = hotspot['hotspot_mask']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from matplotlib import rcParams, font_manager

# --- Set default font to Times New Roman ---
rcParams['font.family'] = 'Times New Roman'

fig, ax = plt.subplots(
    1, 1,
    subplot_kw={'projection': ccrs.PlateCarree()},
    figsize=(10, 7)
)

ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
ax.coastlines(resolution='10m', linewidth=1)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# Extract lat/lon
lat_vals = hotspot_da['latitude'].values
lon_vals = hotspot_da['longitude'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

# Mask only True values
mask = hotspot_da.values == True

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]

# Plot hotspot points
ax.scatter(
    lon_pts,
    lat_pts,
    color='red',
    s=5,
    marker='o',
    edgecolors='black',
    linewidths=0.3,
    transform=ccrs.PlateCarree(),
    zorder=5
)

# Title
ax.set_title(
    'Adaptation relevant Hotspots (1980–2019)',
    fontsize=14,
    weight='bold',
    family='Times New Roman'
)
# ---------------------------
# GRIDLINES
# ---------------------------
gl = ax.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)
# Legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           label='Hotspots',
           markerfacecolor='red',
           markeredgecolor='black',
           markersize=8)
]

ax.legend(
    handles=legend_elements,
    loc='lower left',
    frameon=True,
    prop=font_manager.FontProperties(family='Times New Roman', size=14, weight='bold')
)

print("Number of hotspot grid cells:", np.sum(mask))

# Save
plt.savefig(
    r"D:\IITD\Doctoral Thesis\Work and Progress ppts\nature communications\Plots_final\Adaptation_relevant_hotspots_1980_2019.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Mean TCR over each climate zone

In [ ]:
import xarray as xr

koppen = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\RobustKGMaps.nc")
print(koppen)

In [ ]:
print(koppen['MasterMap1'].shape)
print(trigger_rate_significant_da_1.shape)

In [ ]:
tcr_decades = {
    "1980-1989": trigger_rate_significant_da_1,
    "1990-1999": trigger_rate_significant_da_2,
    "2000-2009": trigger_rate_significant_da_3,
    "2010-2019": trigger_rate_significant_da_4
}

In [ ]:
koppen_da = koppen['MasterMap1']

In [ ]:
import numpy as np

zones = np.unique(koppen_da.values)
zones = zones[~np.isnan(zones)]
zones = zones.astype(int)

In [ ]:
koppen_da 

In [ ]:
np.unique(koppen_da.values)

In [ ]:
for zone in zones:
    count = np.sum(koppen_da.values == zone)
    print(zone, count)

In [ ]:
zone_names_full = {
1: "Af – Tropical Rainforest",
2: "Am – Tropical Monsoon",
3: "Aw – Tropical Savanna",

4: "BSh – Hot Semi-arid (Steppe)",
5: "BSk – Cold Semi-arid (Steppe)",
6: "BWh – Hot Desert",
7: "BWk – Cold Desert",

8: "Cfa – Humid Subtropical",
9: "Cfb – Marine West Coast",
10: "Cfc – Subpolar Oceanic",

11: "Csa – Mediterranean (Hot Summer)",
12: "Csb – Mediterranean (Warm Summer)",
13: "Csc – Mediterranean (Cold Summer)",

14: "Cwa – Humid Subtropical (Dry Winter)",
15: "Cwb – Subtropical Highland (Dry Winter)",
16: "Cwc – Subpolar Highland (Dry Winter)",

17: "Dfa – Humid Continental (Hot Summer)",
18: "Dfb – Humid Continental (Warm Summer)",
19: "Dfc – Subarctic",
20: "Dfd – Extremely Cold Subarctic",

21: "Dsa – Continental Mediterranean (Hot Summer)",
22: "Dsb – Continental Mediterranean (Warm Summer)",
23: "Dsc – Continental Mediterranean (Cold Summer)",
24: "Dsd – Continental Mediterranean (Very Cold)",

25: "Dwa – Humid Continental (Dry Winter Hot Summer)",
26: "Dwb – Humid Continental (Dry Winter Warm Summer)",
27: "Dwc – Subarctic (Dry Winter)",
28: "Dwd – Extremely Cold Subarctic (Dry Winter)",

29: "EF – Ice Cap",
30: "ET – Tundra"
}

In [ ]:
import pandas as pd

results = {}

for zone in zones:
    
    zone_mask = koppen_da.values == zone
    
    results[zone] = {}
    
    for decade, tcr_da in tcr_decades.items():
        
        tcr_vals = tcr_da.values[zone_mask]
        
        results[zone][decade] = np.nanmean(tcr_vals)

In [ ]:
tcr_table = pd.DataFrame(results).T

tcr_table.index = [zone_names_full[z] for z in tcr_table.index]

In [ ]:
tcr_table = tcr_table.reset_index()
tcr_table = tcr_table.rename(columns={'index': 'Climate Zone'})

In [ ]:
tcr_table = tcr_table.reset_index(drop=True)

In [ ]:
tcr_table.style.set_properties(
    subset=['Climate Zone'], 
    **{'text-align': 'left'}
).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]}
])

In [ ]:
tcr_table.to_csv(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\mean_tcr_climatezone.csv", index=False, encoding="utf-8-sig")

# Exploring the Relationship between TCR and transition lag

In [ ]:
lags = range(1,7)   # 0–6 months

In [ ]:
from collections import defaultdict

lag_dicts = {}

for lag in lags:

    lag_dict = defaultdict(list)

    for grid, events in whiplash_dict_c_e.items():

        for event in events:

            if event['lag'] == lag:
                lag_dict[grid].append(event)

    lag_dicts[lag] = lag_dict

In [ ]:
for lag in lags:
    
    count = sum(len(events) for events in lag_dicts[lag].values())
    
    print(f"Lag {lag}: {count} events")

In [ ]:
nT1 = len(times1)
nT2 = len(times2)
nT3 = len(times3)
nT4 = len(times4)

## lag 1

In [ ]:
whiplash_lag1_dec1 = np.full((nT1, nlat, nlon), np.nan)
for (i, j), ev_list in lag_dicts[1].items():
    for ev in ev_list:
        # Get the actual event time range
        ev_start_time = all_times[ev["tc_index"]]
        ev_end_time   = all_times[ev["td_index"]]
        
        # Check if event overlaps with the decade
        if ev_end_time < pd.Timestamp(decade_start1) or ev_start_time > pd.Timestamp(decade_end1):
            continue  # skip events outside this decade

        # Clip event to the decade
        ev_start_time = max(ev_start_time, pd.Timestamp(decade_start1))
        ev_end_time   = min(ev_end_time, pd.Timestamp(decade_end1))
        
        # Convert back to indices relative to decade
        t_start = times1.get_loc(ev_start_time)
        t_end   = times1.get_loc(ev_end_time) + 1

        for t_idx in range(t_start, t_end):
            if np.isnan(whiplash_lag1_dec1[t_idx, i, j]):
                whiplash_lag1_dec1[t_idx, i, j] = ev["score"]
            else:
                whiplash_lag1_dec1[t_idx, i, j] = max(
                    whiplash_lag1_dec1[t_idx, i, j],
                    ev["score"]
                )
whiplash_lag1_dec1_da = xr.DataArray(
    data=whiplash_lag1_dec1,          # numpy array with all values filled
    coords={
        'time': times1,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)
whiplash_month_lag1_dec1 = (whiplash_lag1_dec1_da > 0).astype(int)
stress_month_dec1 = (ws_decade1 < ST20).astype(int)



lat_size = whiplash_month_lag1_dec1.latitude.size
lon_size = whiplash_month_lag1_dec1.longitude.size

def ECA_trigger_grid(X, Y, deltaT=5, tau=0):
    """
    X : 1D numpy array (binary whiplash indicator, 1 during whiplash sequence)
    Y : 1D numpy array (binary water stress indicator, 1 during stress)
    deltaT : maximum allowed delay after whiplash sequence ends (months)
    tau : minimum lag after whiplash sequence ends (default 0)

    Returns:
    r_trig : trigger coincidence rate
    """

    import numpy as np

    # indices where whiplash exists
    tX = np.where(X == 1)[0]
    tY = np.where(Y == 1)[0]

    NX = len(tX)
    NY = len(tY)

    if NX == 0 or NY == 0:
        return np.nan

    # Identify contiguous whiplash sequences
    sequences = []
    start = tX[0]

    for i in range(1, NX):
        if tX[i] != tX[i-1] + 1:
            sequences.append((start, tX[i-1]))
            start = tX[i]

    sequences.append((start, tX[-1]))

    # Check stress AFTER sequence end
    triggered = 0
    total_sequences = len(sequences)

    for seq in sequences:

        tc, td = seq   # td = end of whiplash sequence

        # define allowed window AFTER sequence
        start_window = td + tau
        end_window   = td + tau + deltaT

        # ensure within bounds
        start_window = max(start_window, 0)
        end_window   = min(end_window, len(Y)-1)

        if start_window <= end_window:

            if np.any(Y[start_window:end_window+1] == 1):
                triggered += 1

    r_trig = triggered / total_sequences

    return r_trig
    
trigger_rate_lag1_dec1 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month_lag1_dec1[:, i, j].values
        Y = stress_month_dec1[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_lag1_dec1[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

# convert to xarray DataArray
trigger_rate_lag1_dec1_da = xr.DataArray(
    data=trigger_rate_lag1_dec1,
    coords={'latitude': whiplash_month_lag1_dec1.latitude, 'longitude': whiplash_month_lag1_dec1.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)
import numpy as np

def trigger_rate_significance(X, Y, tau=0, delT=5, n_iter=1000, seed=None):
    """
    X, Y : 1D binary arrays
    tau : lag
    delT : window length
    n_iter : number of Monte Carlo simulations
    Returns: observed rate, p-value
    """
    np.random.seed(seed)
    
    # observed trigger rate
    r_obs = ECA_trigger_grid(X, Y, deltaT=5, tau=0)
    
    # number of events
    NX = X.sum()
    NY = Y.sum()
    
    if NX == 0 or NY == 0:
        return r_obs, np.nan
    
    r_null = []
    T = len(X)
    
    for _ in range(n_iter):
        # randomly place X and Y events in time
        X_rand = np.zeros(T, dtype=int)
        Y_rand = np.zeros(T, dtype=int)
        
        X_idx = np.random.choice(T, NX, replace=False)
        Y_idx = np.random.choice(T, NY, replace=False)
        
        X_rand[X_idx] = 1
        Y_rand[Y_idx] = 1
        
        r_null.append(ECA_trigger_grid(X_rand, Y_rand, tau))
    
    r_null = np.array(r_null)
    
    # p-value: fraction of random trigger rates >= observed
    p_value = (r_null >= r_obs).sum() / n_iter
    
    return r_obs, p_value

p_value_lag1_dec1 = np.full((lat_size, lon_size), np.nan)
trigger_lag1_dec1 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month_lag1_dec1[:, i, j].values
        Y =  stress_month_dec1[:, i, j].values
        
        r_obs_lag1_1, p_val_lag1_1 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_lag1_dec1[i, j] = r_obs_lag1_1
        p_value_lag1_dec1[i, j] = p_val_lag1_1

# convert to xarray DataArray
p_value_lag1_dec1_da = xr.DataArray(
    data= p_value_lag1_dec1,
    coords={'latitude': whiplash_month_lag1_dec1.latitude, 'longitude': whiplash_month_lag1_dec1.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

trigger_rate_significant_lag1_dec1_da= trigger_rate_lag1_dec1_da.where(p_value_lag1_dec1_da < 0.05)

In [ ]:
whiplash_lag1_dec1 = np.full((nT1, nlat, nlon), np.nan)
whiplash_lag1_dec2 = np.full((nT2, nlat, nlon), np.nan)
whiplash_lag1_dec3 = np.full((nT3, nlat, nlon), np.nan)
whiplash_lag1_dec4 = np.full((nT4, nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in lag_dicts[1].items():
    for ev in ev_list:
        # Get the actual event time range
        ev_start_time = all_times[ev["tc_index"]]
        ev_end_time   = all_times[ev["td_index"]]
        
        # Check if event overlaps with the decade
        if ev_end_time < pd.Timestamp(decade_start1) or ev_start_time > pd.Timestamp(decade_end1):
            continue  # skip events outside this decade

        # Clip event to the decade
        ev_start_time = max(ev_start_time, pd.Timestamp(decade_start1))
        ev_end_time   = min(ev_end_time, pd.Timestamp(decade_end1))
        
        # Convert back to indices relative to decade
        t_start = times1.get_loc(ev_start_time)
        t_end   = times1.get_loc(ev_end_time) + 1

        for t_idx in range(t_start, t_end):
            if np.isnan(whiplash_lag1_dec1[t_idx, i, j]):
                whiplash_lag1_dec1[t_idx, i, j] = ev["score"]
            else:
                whiplash_lag1_dec1[t_idx, i, j] = max(
                    whiplash_lag1_dec1[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
for (i, j), ev_list in lag_dicts[1].items():
    for ev in ev_list:
        # Get the actual event time range
        ev_start_time = all_times[ev["tc_index"]]
        ev_end_time   = all_times[ev["td_index"]]
        
        # Check if event overlaps with the decade
        if ev_end_time < pd.Timestamp(decade_start2) or ev_start_time > pd.Timestamp(decade_end2):
            continue  # skip events outside this decade

        # Clip event to the decade
        ev_start_time = max(ev_start_time, pd.Timestamp(decade_start2))
        ev_end_time   = min(ev_end_time, pd.Timestamp(decade_end2))
        
        # Convert back to indices relative to decade
        t_start = times2.get_loc(ev_start_time)
        t_end   = times2.get_loc(ev_end_time) + 1

        for t_idx in range(t_start, t_end):
            if np.isnan(whiplash_lag1_dec2[t_idx, i, j]):
                whiplash_lag1_dec2[t_idx, i, j] = ev["score"]
            else:
                whiplash_lag1_dec2[t_idx, i, j] = max(
                    whiplash_lag1_dec2[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
for (i, j), ev_list in lag_dicts[1].items():
    for ev in ev_list:
        # Get the actual event time range
        ev_start_time = all_times[ev["tc_index"]]
        ev_end_time   = all_times[ev["td_index"]]
        
        # Check if event overlaps with the decade
        if ev_end_time < pd.Timestamp(decade_start3) or ev_start_time > pd.Timestamp(decade_end3):
            continue  # skip events outside this decade

        # Clip event to the decade
        ev_start_time = max(ev_start_time, pd.Timestamp(decade_start3))
        ev_end_time   = min(ev_end_time, pd.Timestamp(decade_end3))
        
        t_start = times3.get_indexer([ev_start_time], method="nearest")[0]
        t_end   = times3.get_indexer([ev_end_time], method="nearest")[0] + 1

        for t_idx in range(t_start, t_end):
            if np.isnan(whiplash_lag1_dec3[t_idx, i, j]):
                whiplash_lag1_dec3[t_idx, i, j] = ev["score"]
            else:
                whiplash_lag1_dec3[t_idx, i, j] = max(
                    whiplash_lag1_dec3[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
for (i, j), ev_list in lag_dicts[1].items():
    for ev in ev_list:
        # Get the actual event time range
        ev_start_time = all_times[ev["tc_index"]]
        ev_end_time   = all_times[ev["td_index"]]
        
        # Check if event overlaps with the decade
        if ev_end_time < pd.Timestamp(decade_start4) or ev_start_time > pd.Timestamp(decade_end4):
            continue  # skip events outside this decade

        # Clip event to the decade
        ev_start_time = max(ev_start_time, pd.Timestamp(decade_start4))
        ev_end_time   = min(ev_end_time, pd.Timestamp(decade_end4))
        
        t_start = times4.get_indexer([ev_start_time], method="nearest")[0]
        t_end   = times4.get_indexer([ev_end_time], method="nearest")[0] + 1

        for t_idx in range(t_start, t_end):
            if np.isnan(whiplash_lag1_dec4[t_idx, i, j]):
                whiplash_lag1_dec4[t_idx, i, j] = ev["score"]
            else:
                whiplash_lag1_dec4[t_idx, i, j] = max(
                    whiplash_lag1_dec4[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
whiplash_lag1_dec1_da = xr.DataArray(
    data=whiplash_lag1_dec1,          # numpy array with all values filled
    coords={
        'time': times1,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
whiplash_lag1_dec2_da = xr.DataArray(
    data=whiplash_lag1_dec2,          # numpy array with all values filled
    coords={
        'time': times2,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
whiplash_lag1_dec3_da = xr.DataArray(
    data=whiplash_lag1_dec3,          # numpy array with all values filled
    coords={
        'time': times3,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
whiplash_lag1_dec4_da = xr.DataArray(
    data=whiplash_lag1_dec4,          # numpy array with all values filled
    coords={
        'time': times4,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
whiplash_month_lag1_dec1 = (whiplash_lag1_dec1_da > 0).astype(int)
stress_month_dec1 = (ws_decade1 < ST20).astype(int)

In [ ]:
whiplash_month_lag1_dec2 = (whiplash_lag1_dec2_da > 0).astype(int)
stress_month_dec2 = (ws_decade2 < ST20).astype(int)

In [ ]:
whiplash_month_lag1_dec3 = (whiplash_lag1_dec3_da > 0).astype(int)
stress_month_dec3 = (ws_decade3 < ST20).astype(int)

In [ ]:
whiplash_month_lag1_dec4 = (whiplash_lag1_dec4_da > 0).astype(int)
stress_month_dec4 = (ws_decade4 < ST20).astype(int)

In [ ]:
lat_size = whiplash_month_lag1_dec1.latitude.size
lon_size = whiplash_month_lag1_dec1.longitude.size

trigger_rate_lag1_dec1 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month_lag1_dec1[:, i, j].values
        Y = stress_month_dec1[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_lag1_dec1[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
lat_size = whiplash_month_lag1_dec2.latitude.size
lon_size = whiplash_month_lag1_dec2.longitude.size

trigger_rate_lag1_dec2 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month_lag1_dec2[:, i, j].values
        Y = stress_month_dec2[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_lag1_dec2[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
lat_size = whiplash_month_lag1_dec3.latitude.size
lon_size = whiplash_month_lag1_dec3.longitude.size

trigger_rate_lag1_dec3 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month_lag1_dec3[:, i, j].values
        Y = stress_month_dec3[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_lag1_dec3[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
lat_size = whiplash_month_lag1_dec4.latitude.size
lon_size = whiplash_month_lag1_dec4.longitude.size

trigger_rate_lag1_dec4 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month_lag1_dec4[:, i, j].values
        Y = stress_month_dec4[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_lag1_dec4[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
# convert to xarray DataArray
trigger_rate_lag1_dec1_da = xr.DataArray(
    data=trigger_rate_lag1_dec1,
    coords={'latitude': whiplash_month_lag1_dec1.latitude, 'longitude': whiplash_month_lag1_dec1.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
# convert to xarray DataArray
trigger_rate_lag1_dec2_da = xr.DataArray(
    data=trigger_rate_lag1_dec2,
    coords={'latitude': whiplash_month_lag1_dec1.latitude, 'longitude': whiplash_month_lag1_dec1.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
# convert to xarray DataArray
trigger_rate_lag1_dec3_da = xr.DataArray(
    data=trigger_rate_lag1_dec3,
    coords={'latitude': whiplash_month_lag1_dec1.latitude, 'longitude': whiplash_month_lag1_dec1.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
# convert to xarray DataArray
trigger_rate_lag1_dec4_da = xr.DataArray(
    data=trigger_rate_lag1_dec4,
    coords={'latitude': whiplash_month_lag1_dec1.latitude, 'longitude': whiplash_month_lag1_dec1.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
p_value_lag1_dec1 = np.full((lat_size, lon_size), np.nan)
trigger_lag1_dec1 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month_lag1_dec1[:, i, j].values
        Y =  stress_month_dec1[:, i, j].values
        
        r_obs_lag1_1, p_val_lag1_1 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_lag1_dec1[i, j] = r_obs_lag1_1
        p_value_lag1_dec1[i, j] = p_val_lag1_1

In [ ]:
p_value_lag1_dec2 = np.full((lat_size, lon_size), np.nan)
trigger_lag1_dec2 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month_lag1_dec2[:, i, j].values
        Y =  stress_month_dec2[:, i, j].values
        
        r_obs_lag1_2, p_val_lag1_2 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_lag1_dec2[i, j] = r_obs_lag1_2
        p_value_lag1_dec2[i, j] = p_val_lag1_2

In [ ]:
p_value_lag1_dec3 = np.full((lat_size, lon_size), np.nan)
trigger_lag1_dec3 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month_lag1_dec3[:, i, j].values
        Y =  stress_month_dec3[:, i, j].values
        
        r_obs_lag1_3, p_val_lag1_3 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_lag1_dec3[i, j] = r_obs_lag1_3
        p_value_lag1_dec3[i, j] = p_val_lag1_3

In [ ]:
p_value_lag1_dec4 = np.full((lat_size, lon_size), np.nan)
trigger_lag1_dec4 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month_lag1_dec4[:, i, j].values
        Y =  stress_month_dec4[:, i, j].values
        
        r_obs_lag1_4, p_val_lag1_4 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_lag1_dec4[i, j] = r_obs_lag1_4
        p_value_lag1_dec4[i, j] = p_val_lag1_4

In [ ]:
# convert to xarray DataArray
p_value_lag1_dec1_da = xr.DataArray(
    data= p_value_lag1_dec1,
    coords={'latitude': whiplash_month_lag1_dec1.latitude, 'longitude': whiplash_month_lag1_dec1.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

## Decadal Lag-wise TCR

In [ ]:
decades = {
    "dec1": ("1980-01-01", "1989-12-01", times1),
    "dec2": ("1990-01-01", "1999-12-01", times2),
    "dec3": ("2000-01-01", "2009-12-01", times3),
    "dec4": ("2010-01-01", "2019-12-01", times4)
}

In [ ]:
whiplash_data = {}

for lag in range(1,7):
    
    for dec in range(1,5):
        
        print(f"Lag {lag}, Decade {dec}")
        
        lag_dict = lag_dicts[lag]
        
        # choose decade variables
        if dec == 1:
            decade_start = decade_start1
            decade_end = decade_end1
            times = times1
        elif dec == 2:
            decade_start = decade_start2
            decade_end = decade_end2
            times = times2
        elif dec == 3:
            decade_start = decade_start3
            decade_end = decade_end3
            times = times3
        else:
            decade_start = decade_start4
            decade_end = decade_end4
            times = times4
        
        nT = len(times)
        
        arr = np.full((nT,nlat,nlon),np.nan)
        
        for (i,j), ev_list in lag_dict.items():
            
            for ev in ev_list:
                
                ev_start = all_times[ev["tc_index"]]
                ev_end   = all_times[ev["td_index"]]
                
                if ev_end < pd.Timestamp(decade_start) or ev_start > pd.Timestamp(decade_end):
                    continue
                
                ev_start = max(ev_start, pd.Timestamp(decade_start))
                ev_end   = min(ev_end, pd.Timestamp(decade_end))
                
                t_start = times.searchsorted(ev_start)
                t_end   = times.searchsorted(ev_end) + 1
                
                for t in range(t_start,t_end):
                    
                    if np.isnan(arr[t,i,j]):
                        arr[t,i,j] = ev["score"]
                    else:
                        arr[t,i,j] = max(arr[t,i,j], ev["score"])
        
        
        whiplash_data[(lag,dec)] = arr

trigger_rates = {}

for lag in range(1,7):
    
    for dec in range(1,5):
        
        print(f"Trigger rate: lag {lag}, decade {dec}")
        
        arr = whiplash_data[(lag,dec)]
        
        whiplash_month = (arr > 0).astype(int)
        
        if dec == 1:
            stress = (ws_decade1 < ST20).astype(int)
        elif dec == 2:
            stress = (ws_decade2 < ST20).astype(int)
        elif dec == 3:
            stress = (ws_decade3 < ST20).astype(int)
        else:
            stress = (ws_decade4 < ST20).astype(int)
        
        
        trigger = np.full((nlat,nlon),np.nan)
        
        for i in range(nlat):
            for j in range(nlon):
                
                X = whiplash_month[:,i,j]
                Y = stress[:,i,j]
                
                trigger[i,j] = ECA_trigger_grid(X,Y,deltaT=3,tau=0)
        
        
        trigger_rates[(lag,dec)] = trigger

def trigger_rate_significance(X, Y, tau=0, delT=5, n_iter=1000, seed=None):
    """
    X, Y : 1D binary arrays
    tau : lag
    delT : window length
    n_iter : number of Monte Carlo simulations
    Returns: observed rate, p-value
    """
    np.random.seed(seed)
    
    # observed trigger rate
    r_obs = ECA_trigger_grid(X, Y, deltaT=5, tau=0)
    
    # number of events
    NX = int(np.nansum(X))
    NY = int(np.nansum(Y))
    
    if NX == 0 or NY == 0:
        return r_obs, np.nan
    
    r_null = []
    T = len(X)
    
    for _ in range(n_iter):
        # randomly place X and Y events in time
        X_rand = np.zeros(T, dtype=int)
        Y_rand = np.zeros(T, dtype=int)
        
        X_idx = np.random.choice(T, NX, replace=False)
        Y_idx = np.random.choice(T, NY, replace=False)
        
        X_rand[X_idx] = 1
        Y_rand[Y_idx] = 1
        
        r_null.append(ECA_trigger_grid(X_rand, Y_rand, tau))
    
    r_null = np.array(r_null)
    
    # p-value: fraction of random trigger rates >= observed
    p_value = (r_null >= r_obs).sum() / n_iter
    
    return r_obs, p_value


p_values = {}

for lag in range(1,7):
    
    for dec in range(1,5):
        
        print(f"Significance: lag {lag}, decade {dec}")
        
        arr = whiplash_data[(lag,dec)]
        whiplash_month = (arr > 0).astype(int)
        
        if dec == 1:
            stress = (ws_decade1 < ST20).astype(int)
        elif dec == 2:
            stress = (ws_decade2 < ST20).astype(int)
        elif dec == 3:
            stress = (ws_decade3 < ST20).astype(int)
        else:
            stress = (ws_decade4 < ST20).astype(int)
        
        p_arr = np.full((nlat,nlon),np.nan)
        
        for i in range(nlat):
            for j in range(nlon):
                
                X = whiplash_month[:,i,j]
                Y = stress[:,i,j]
                
                _, p = trigger_rate_significance(
                    X,Y,tau=0,delT=3,n_iter=500
                )
                
                p_arr[i,j] = p
        
        p_values[(lag,dec)] = p_arr

import xarray as xr

trigger_rate_significant = {}

for lag in range(1, 7):
    for dec in range(1, 5):
        # Convert arrays to DataArrays
        tr_da = xr.DataArray(
            trigger_rates[(lag, dec)],
            coords={'latitude': lat_values, 'longitude': lon_values},
            dims=('latitude', 'longitude')
        )
        p_da = xr.DataArray(
            p_values[(lag, dec)],
            coords={'latitude': lat_values, 'longitude': lon_values},
            dims=('latitude', 'longitude')
        )
        
        # Keep only significant trigger rates
        trigger_rate_significant[(lag, dec)] = tr_da.where(p_da < 0.05)

import xarray as xr
import numpy as np

# assume trigger_rate_significant is your dict with keys (lag, dec)
lags = range(1, 7)
decades = range(1, 5)

# get latitude and longitude from any of your DataArrays
lat = trigger_rate_significant[(1, 1)].latitude
lon = trigger_rate_significant[(1, 1)].longitude

# initialize empty 4D array
combined_array = np.full((len(lags), len(decades), len(lat), len(lon)), np.nan)

# fill it
for i, lag in enumerate(lags):
    for j, dec in enumerate(decades):
        combined_array[i, j, :, :] = trigger_rate_significant[(lag, dec)].values

# create xarray DataArray
combined_da = xr.DataArray(
    data=combined_array,
    coords={
        'lag': lags,
        'decade': decades,
        'latitude': lat,
        'longitude': lon
    },
    dims=('lag', 'decade', 'latitude', 'longitude'),
    name='trigger_rate_significant',
    attrs={
        'long_name': 'Significant trigger coincidence rate between whiplash and water stress',
        'units': 'fraction (0-1)'
    }
)

# save to NetCDF
combined_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_significant_4D.nc")

In [ ]:
whiplash_data = {}

for lag in range(1,7):
    
    for dec in range(1,5):
        
        print(f"Lag {lag}, Decade {dec}")
        
        lag_dict = lag_dicts[lag]
        
        # choose decade variables
        if dec == 1:
            decade_start = decade_start1
            decade_end = decade_end1
            times = times1
        elif dec == 2:
            decade_start = decade_start2
            decade_end = decade_end2
            times = times2
        elif dec == 3:
            decade_start = decade_start3
            decade_end = decade_end3
            times = times3
        else:
            decade_start = decade_start4
            decade_end = decade_end4
            times = times4
        
        nT = len(times)
        
        arr = np.full((nT,nlat,nlon),np.nan)
        
        for (i,j), ev_list in lag_dict.items():
            
            for ev in ev_list:
                
                ev_start = all_times[ev["tc_index"]]
                ev_end   = all_times[ev["td_index"]]
                
                if ev_end < pd.Timestamp(decade_start) or ev_start > pd.Timestamp(decade_end):
                    continue
                
                ev_start = max(ev_start, pd.Timestamp(decade_start))
                ev_end   = min(ev_end, pd.Timestamp(decade_end))
                
                t_start = times.searchsorted(ev_start)
                t_end   = times.searchsorted(ev_end) + 1
                
                for t in range(t_start,t_end):
                    
                    if np.isnan(arr[t,i,j]):
                        arr[t,i,j] = ev["score"]
                    else:
                        arr[t,i,j] = max(arr[t,i,j], ev["score"])
        
        
        whiplash_data[(lag,dec)] = arr

In [ ]:
trigger_rates = {}

for lag in range(1,7):
    
    for dec in range(1,5):
        
        print(f"Trigger rate: lag {lag}, decade {dec}")
        
        arr = whiplash_data[(lag,dec)]
        
        whiplash_month = (arr > 0).astype(int)
        
        if dec == 1:
            stress = (ws_decade1 < ST20).astype(int)
        elif dec == 2:
            stress = (ws_decade2 < ST20).astype(int)
        elif dec == 3:
            stress = (ws_decade3 < ST20).astype(int)
        else:
            stress = (ws_decade4 < ST20).astype(int)
        
        
        trigger = np.full((nlat,nlon),np.nan)
        
        for i in range(nlat):
            for j in range(nlon):
                
                X = whiplash_month[:,i,j]
                Y = stress[:,i,j]
                
                trigger[i,j] = ECA_trigger_grid(X,Y,deltaT=3,tau=0)
        
        
        trigger_rates[(lag,dec)] = trigger

In [ ]:
trigger_rates

In [ ]:
def trigger_rate_significance(X, Y, tau=0, delT=5, n_iter=1000, seed=None):
    """
    X, Y : 1D binary arrays
    tau : lag
    delT : window length
    n_iter : number of Monte Carlo simulations
    Returns: observed rate, p-value
    """
    np.random.seed(seed)
    
    # observed trigger rate
    r_obs = ECA_trigger_grid(X, Y, deltaT=5, tau=0)
    
    # number of events
    NX = int(np.nansum(X))
    NY = int(np.nansum(Y))
    
    if NX == 0 or NY == 0:
        return r_obs, np.nan
    
    r_null = []
    T = len(X)
    
    for _ in range(n_iter):
        # randomly place X and Y events in time
        X_rand = np.zeros(T, dtype=int)
        Y_rand = np.zeros(T, dtype=int)
        
        X_idx = np.random.choice(T, NX, replace=False)
        Y_idx = np.random.choice(T, NY, replace=False)
        
        X_rand[X_idx] = 1
        Y_rand[Y_idx] = 1
        
        r_null.append(ECA_trigger_grid(X_rand, Y_rand, tau))
    
    r_null = np.array(r_null)
    
    # p-value: fraction of random trigger rates >= observed
    p_value = (r_null >= r_obs).sum() / n_iter
    
    return r_obs, p_value

In [ ]:
p_values = {}

for lag in range(1,7):
    
    for dec in range(1,5):
        
        print(f"Significance: lag {lag}, decade {dec}")
        
        arr = whiplash_data[(lag,dec)]
        whiplash_month = (arr > 0).astype(int)
        
        if dec == 1:
            stress = (ws_decade1 < ST20).astype(int)
        elif dec == 2:
            stress = (ws_decade2 < ST20).astype(int)
        elif dec == 3:
            stress = (ws_decade3 < ST20).astype(int)
        else:
            stress = (ws_decade4 < ST20).astype(int)
        
        p_arr = np.full((nlat,nlon),np.nan)
        
        for i in range(nlat):
            for j in range(nlon):
                
                X = whiplash_month[:,i,j]
                Y = stress[:,i,j]
                
                _, p = trigger_rate_significance(
                    X,Y,tau=0,delT=3,n_iter=500
                )
                
                p_arr[i,j] = p
        
        p_values[(lag,dec)] = p_arr

In [ ]:
p_values

In [ ]:
import xarray as xr

trigger_rate_significant = {}

for lag in range(1, 7):
    for dec in range(1, 5):
        # Convert arrays to DataArrays
        tr_da = xr.DataArray(
            trigger_rates[(lag, dec)],
            coords={'latitude': lat_values, 'longitude': lon_values},
            dims=('latitude', 'longitude')
        )
        p_da = xr.DataArray(
            p_values[(lag, dec)],
            coords={'latitude': lat_values, 'longitude': lon_values},
            dims=('latitude', 'longitude')
        )
        
        # Keep only significant trigger rates
        trigger_rate_significant[(lag, dec)] = tr_da.where(p_da < 0.05)

In [ ]:
trigger_rate_significant

In [ ]:
import xarray as xr
import numpy as np

# assume trigger_rate_significant is your dict with keys (lag, dec)
lags = range(1, 7)
decades = range(1, 5)

# get latitude and longitude from any of your DataArrays
lat = trigger_rate_significant[(1, 1)].latitude
lon = trigger_rate_significant[(1, 1)].longitude

# initialize empty 4D array
combined_array = np.full((len(lags), len(decades), len(lat), len(lon)), np.nan)

# fill it
for i, lag in enumerate(lags):
    for j, dec in enumerate(decades):
        combined_array[i, j, :, :] = trigger_rate_significant[(lag, dec)].values

# create xarray DataArray
combined_da = xr.DataArray(
    data=combined_array,
    coords={
        'lag': lags,
        'decade': decades,
        'latitude': lat,
        'longitude': lon
    },
    dims=('lag', 'decade', 'latitude', 'longitude'),
    name='trigger_rate_significant',
    attrs={
        'long_name': 'Significant trigger coincidence rate between whiplash and water stress',
        'units': 'fraction (0-1)'
    }
)

In [ ]:
combined_da

In [ ]:
# save to NetCDF
combined_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_significant_4D.nc")

## Lag-wise TCR

In [ ]:
import xarray as xr

In [ ]:
import json
from collections import defaultdict

with open(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\filtered_whiplash_events.json", "r") as f:
    loaded_dict3 = json.load(f)

In [ ]:
whiplash_dict_c_e = defaultdict(list, {eval(k): v for k, v in loaded_dict3.items()})

In [ ]:
lags = range(1,7)   # 0–6 months
from collections import defaultdict

lag_dicts = {}

for lag in lags:

    lag_dict = defaultdict(list)

    for grid, events in whiplash_dict_c_e.items():

        for event in events:

            if event['lag'] == lag:
                lag_dict[grid].append(event)

    lag_dicts[lag] = lag_dict

In [ ]:
whiplash_dict_c_e

In [ ]:
water_stress_ds =xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\Water Stress\water_stress_1980_2019.nc")

In [ ]:
water_stress_da = water_stress_ds["__xarray_dataarray_variable__"]

In [ ]:
# Select baseline period
baseline_stress = water_stress_da.sel(time=slice("1995-01-01", "2014-12-31"))

In [ ]:
# Compute 90th percentile per grid point
ST20 = baseline_stress.quantile(0.20, dim="time")

# P90 dims: (lat, lon)
print(ST20.dims)

In [ ]:
stress_full = (water_stress_da < ST20).astype(int)

In [ ]:
times = pd.date_range("1980-01-01", "2019-12-01", freq="MS")  # 480 months

In [ ]:
nT = len(times)


In [ ]:
whiplash_data = {}

for lag in range(1,7):
    
    print(f"Lag {lag}")
    
    lag_dict = lag_dicts[lag]
    
    arr = np.full((nT, nlat, nlon), np.nan)
    
    for (i,j), ev_list in lag_dict.items():
        
        for ev in ev_list:
            
            ev_start = times[ev["tc_index"]]
            ev_end   = times[ev["td_index"]]
            
            # map to full timeline
            t_start = times.searchsorted(ev_start)
            t_end   = times.searchsorted(ev_end) + 1
            
            for t in range(t_start, t_end):
                
                if np.isnan(arr[t,i,j]):
                    arr[t,i,j] = ev["score"]
                else:
                    arr[t,i,j] = max(arr[t,i,j], ev["score"])
    
    whiplash_data[lag] = arr

In [ ]:
for lag in range(1,7):
    print(f"Lag {lag}: total events =",
          sum(len(v) for v in lag_dicts[lag].values()))

In [ ]:
for lag in range(1,7):
    
    unique_lags = set()
    
    for ev_list in lag_dicts[lag].values():
        for ev in ev_list:
            unique_lags.add(ev['lag'])
    
    print(f"Lag {lag}: contains lags ->", unique_lags)

In [ ]:
for lag in range(2,7):
    
    diff = np.nanmean(np.abs(
        whiplash_data[lag] - whiplash_data[1]
    ))
    
    print(f"Difference lag {lag} vs lag 1:", diff)

In [ ]:
def ECA_trigger_grid(X, Y, deltaT=5, tau=0):
    """
    X : 1D numpy array (binary whiplash indicator, 1 during whiplash sequence)
    Y : 1D numpy array (binary water stress indicator, 1 during stress)
    deltaT : maximum allowed delay after whiplash sequence ends (months)
    tau : minimum lag after whiplash sequence ends (default 0)

    Returns:
    r_trig : trigger coincidence rate
    """

    import numpy as np

    # indices where whiplash exists
    tX = np.where(X == 1)[0]
    tY = np.where(Y == 1)[0]

    NX = len(tX)
    NY = len(tY)

    if NX == 0 or NY == 0:
        return np.nan

    # Identify contiguous whiplash sequences
    sequences = []
    start = tX[0]

    for i in range(1, NX):
        if tX[i] != tX[i-1] + 1:
            sequences.append((start, tX[i-1]))
            start = tX[i]

    sequences.append((start, tX[-1]))

    # Check stress AFTER sequence end
    triggered = 0
    total_sequences = len(sequences)

    for seq in sequences:

        tc, td = seq   # td = end of whiplash sequence

        # define allowed window AFTER sequence
        start_window = td + tau
        end_window   = td + tau + deltaT

        # ensure within bounds
        start_window = max(start_window, 0)
        end_window   = min(end_window, len(Y)-1)

        if start_window <= end_window:

            if np.any(Y[start_window:end_window+1] == 1):
                triggered += 1

    r_trig = triggered / total_sequences

    return r_trig

In [ ]:
trigger_rates = {}

for lag in range(1,7):
    
    print(f"Trigger rate: lag {lag}")
    
    arr = whiplash_data[lag]
    whiplash_month = (arr > 0).astype(int)
    
    trigger = np.full((nlat,nlon), np.nan)
    
    for i in range(nlat):
        for j in range(nlon):
            
            X = whiplash_month[:,i,j]
            Y = stress_full[:,i,j]
            
            trigger[i,j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)
    
    trigger_rates[lag] = trigger

In [ ]:
trigger_rates

In [ ]:
# Convert numpy arrays to xarray DataArrays
trigger_rates_xr = {
    lag: xr.DataArray(
        data=arr,
        dims=['latitude', 'longitude'],
        coords={
            'latitude': lat_values,
            'longitude': lon_values
        },
        name=f'trigger_rate_lag_{lag}'
    )
    for lag, arr in trigger_rates.items()
}

In [ ]:
import xarray as xr

trigger_rates_da = {}

for lag in range(1, 7):
    trigger_rates_da[lag] = xr.DataArray(
        trigger_rates[lag],
        coords={
            'latitude': lat_values,
            'longitude': lon_values
        },
        dims=('latitude', 'longitude'),
        name=f'trigger_rate_lag_{lag}'
    )

# Optional: replace the original dictionary
#trigger_rates = trigger_rates_da

In [ ]:
trigger_rates_da 

In [ ]:
ds_trig= xr.Dataset({
    f"class_{k}": da
    for k, da in trigger_rates_da.items()
})

ds_trig.to_netcdf(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\trigger_rate_all_1980_2019_3_lag_based.nc"
)

In [ ]:
def trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=1000, seed=None):
    """
    X, Y : 1D binary arrays
    tau : lag
    delT : window length
    n_iter : number of Monte Carlo simulations
    Returns: observed rate, p-value
    """
    np.random.seed(seed)
    
    # observed trigger rate
    r_obs = ECA_trigger_grid(X, Y, deltaT=3, tau=0)
    
    # number of events
    NX = int(np.nansum(X))
    NY = int(np.nansum(Y))
    
    if NX == 0 or NY == 0:
        return r_obs, np.nan
    
    r_null = []
    T = len(X)
    
    for _ in range(n_iter):
        # randomly place X and Y events in time
        X_rand = np.zeros(T, dtype=int)
        Y_rand = np.zeros(T, dtype=int)
        
        X_idx = np.random.choice(T, NX, replace=False)
        Y_idx = np.random.choice(T, NY, replace=False)
        
        X_rand[X_idx] = 1
        Y_rand[Y_idx] = 1
        
        r_null.append(ECA_trigger_grid(X_rand, Y_rand, tau))
    
    r_null = np.array(r_null)
    
    # p-value: fraction of random trigger rates >= observed
    p_value = (r_null >= r_obs).sum() / n_iter
    
    return r_obs, p_value

In [ ]:
p_values = {}

for lag in range(1,7):
    
    print(f"Significance: lag {lag}")
    
    arr = whiplash_data[lag]
    whiplash_month = (arr > 0).astype(int)
    
    p_arr = np.full((nlat,nlon), np.nan)
    
    for i in range(nlat):
        for j in range(nlon):
            
            X = whiplash_month[:,i,j]
            Y = stress_full[:,i,j]
            
            _, p = trigger_rate_significance(
                X, Y, tau=0, delT=3, n_iter=500
            )
            
            p_arr[i,j] = p
    
    p_values[lag] = p_arr

In [ ]:
trigger_rate_significant = {}

for lag in range(1,7):
    
    tr_da = xr.DataArray(
        trigger_rates[lag],
        coords={'latitude': lat_values, 'longitude': lon_values},
        dims=('latitude','longitude')
    )
    
    p_da = xr.DataArray(
        p_values[lag],
        coords={'latitude': lat_values, 'longitude': lon_values},
        dims=('latitude','longitude')
    )
    
    trigger_rate_significant[lag] = tr_da.where(p_da < 0.05)

In [ ]:
trigger_rate_significant

In [ ]:
ds_sig = xr.Dataset({
    f"class_{k}": da
    for k, da in trigger_rate_significant.items()
})

ds_sig.to_netcdf(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\trigger_rate_significant_1980_2019_3_lag_based.nc"
)

In [ ]:
import xarray as xr

ds_sig = xr.open_dataset(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\trigger_rate_significant_1980_2019_3_lag_based.nc"
)

trigger_rate_significant = {
    int(var.split('_')[1]): ds_sig[var]
    for var in ds_sig.data_vars
}

In [ ]:
ds_trig = xr.open_dataset(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\trigger_rate_all_1980_2019_3_lag_based.nc"
)

trigger_rate= {
    int(var.split('_')[1]): ds_trig[var]
    for var in ds_trig.data_vars
}

In [ ]:
trigger_rate_significant

In [ ]:
trigger_rate_significant_da_3_ds = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_all_1980_2019_3.nc")

In [ ]:
import numpy as np

for lag, da in trigger_rate_significant.items():
    
    all_nan = np.isnan(da.values).all()
    
    print(f"Lag {lag}: All NaN = {all_nan}")

In [ ]:
import numpy as np

for lag, da in trigger_rate_significant.items():
    
    vals = da.values
    vals = vals[np.isfinite(vals)]  # remove NaNs
    
    unique_vals = np.unique(vals)
    
    print(f"\nLag {lag}:")
    print("Min:", np.min(vals) if len(vals)>0 else None)
    print("Max:", np.max(vals) if len(vals)>0 else None)
    print("Unique (sample):", unique_vals[:10])  # first 10 values
    print("Count:", len(vals))

In [ ]:
for lag in range(1,7):
    print(f"\nLag {lag}")
    print("Mean ALL:", np.nanmean(trigger_rates[lag]))
    print("Mean SIG:", np.nanmean(trigger_rate_significant[lag].values))

# Climate wise Mean TCR

In [ ]:
import xarray as xr

koppen = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\RobustKGMaps.nc")
print(koppen)

In [ ]:
koppen_da = koppen['MasterMap1']

In [ ]:
import xarray as xr
import numpy as np

import numpy as np

koppen_vals = koppen_da.values

koppen_major = np.full_like(koppen_vals, '', dtype=object)

koppen_major[(koppen_vals >= 1) & (koppen_vals <= 3)] = 'A'
koppen_major[(koppen_vals >= 4) & (koppen_vals <= 7)] = 'B'
koppen_major[(koppen_vals >= 8) & (koppen_vals <= 16)] = 'C'
koppen_major[(koppen_vals >= 17) & (koppen_vals <= 28)] = 'D'
koppen_major[(koppen_vals >= 29) & (koppen_vals <= 30)] = 'E'

In [ ]:
koppen_major

In [ ]:
lags = range(1, 7)
climates = ['A','B','C','D','E']

In [ ]:
import numpy as np
import pandas as pd

heatmap_data = []

for cls in climates:
    
    row = []
    mask = (koppen_major == cls)
    
    for lag in lags:
        
        da = trigger_rates[lag]   # ✅ FIX HERE
        
        vals = da[mask]
        mean_val = np.nanmean(vals)
        
        row.append(mean_val)
    
    heatmap_data.append(row)

In [ ]:
heatmap_data = [
    [0.5192307692307693, 0.3870073891625616, 0.29250423011844334, 0.2966396292004635, 0.261, 0.33585858585858586],
    [0.45271604938271603, 0.4984717727436174, 0.3887902987281869, 0.36748633879781417, 0.33191489361702126, 0.46176470588235297],
    [0.34628378378378377, 0.3762870012870012, 0.35253674727358936, 0.2953020134228188, 0.36774628879892035, 0.452755905511811],
    [0.7650494159928122, 0.6920593931568754, 0.6997687564234327, 0.5615357014711159, 0.5708722741433022, 0.46958637469586373],
    [0.715210355987055, 0.6011560693641619, 0.6591954022988507, 0.6265306122448979, 0.6307471264367817, 0.44370860927152317]
]

In [ ]:
heatmap_data

In [ ]:
heatmap_df = pd.DataFrame(
    heatmap_data,
    index=climates,
    columns=lags
)

In [ ]:
heatmap_df.index = ['Tropical','Arid','Temperate','Cold','Polar']
heatmap_df.columns = [f"Lag {l}" for l in lags]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams

# Font settings
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14
rcParams['font.weight'] = 'bold'

plt.figure(figsize=(10,5))

sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=0.2,
    vmax=0.8
)

# Axis labels
plt.xlabel("Lag (months)", fontweight='bold')   # ✅ updated
plt.ylabel("Climate Zone", fontweight='bold')

# Title
plt.title("Climate-wise Mean TCR across Lags", fontweight='bold')

# ✅ Set tick labels as 1–6
plt.xticks(
    ticks=np.arange(len(lags)) + 0.5,
    labels=[str(l) for l in lags],
    rotation=0,
    fontweight='bold'
)

plt.yticks(rotation=0, fontweight='bold')

plt.tight_layout()

plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\climatewise_mean_tcr_lags.tif",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# --------------------------------------------------
# Köppen 30 climate classes
# --------------------------------------------------
koppen_labels = {
    1: 'Af',  2: 'Am',  3: 'Aw',
    4: 'BWh', 5: 'BWk', 6: 'BSh', 7: 'BSk',
    8: 'Csa', 9: 'Csb', 10: 'Csc',
    11: 'Cwa', 12: 'Cwb', 13: 'Cwc',
    14: 'Cfa', 15: 'Cfb', 16: 'Cfc',
    17: 'Dsa', 18: 'Dsb', 19: 'Dsc', 20: 'Dsd',
    21: 'Dwa', 22: 'Dwb', 23: 'Dwc', 24: 'Dwd',
    25: 'Dfa', 26: 'Dfb', 27: 'Dfc', 28: 'Dfd',
    29: 'ET', 30: 'EF'
}

lags = range(1, 7)

In [ ]:
# --------------------------------------------------
# Compute mean TCR for each climate class and lag
# --------------------------------------------------
heatmap_data = []

for code in sorted(koppen_labels.keys()):
    row = []
    mask = (koppen_vals == code)

    for lag in lags:
        da = trigger_rate[lag].values
        mean_val = np.nanmean(da[mask])
        row.append(mean_val)

    heatmap_data.append(row)


In [ ]:
# --------------------------------------------------
# Create DataFrame
# --------------------------------------------------
heatmap_df = pd.DataFrame(
    heatmap_data,
    index=[koppen_labels[k] for k in sorted(koppen_labels.keys())],
    columns=[f"Lag {l}" for l in lags]
)

print(heatmap_df)

In [ ]:
# --------------------------------------------------
# Plot
# --------------------------------------------------
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 11
rcParams['font.weight'] = 'bold'

plt.figure(figsize=(8, 10))

ax = sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=np.nanmin(heatmap_df.values),
    vmax=np.nanmax(heatmap_df.values),
    linewidths=0.3,
    annot_kws={"size": 9, "weight": "bold"},
    cbar_kws={
        'label': 'Mean TCR',
        'shrink': 1.0,      # full height to match heatmap
        'aspect': 40,       # larger value = thinner colorbar
        'pad': 0.02
    }
)

plt.xlabel("Lag (months)", fontweight='bold')
plt.ylabel("Köppen Climate Class", fontweight='bold')
plt.title("Köppen Climate-wise Mean TCR across Lags", fontweight='bold')

plt.xticks(
    ticks=np.arange(len(lags)) + 0.5,
    labels=[str(l) for l in lags],
    rotation=0,
    fontweight='bold'
)

plt.yticks(rotation=0, fontweight='bold')

# Colorbar formatting
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=10)
cbar.set_label('Mean TCR', fontsize=11, fontweight='bold')

plt.tight_layout()

plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\extra figures from compri\koppen_30zone_mean_tcr_lags.tiff",
    dpi=600,
    bbox_inches='tight'
)
plt.show()

In [ ]:
lags = range(1,7)   # 0–6 months
from collections import defaultdict

lag_dicts = {}

for lag in lags:

    lag_dict = defaultdict(list)

    for grid, events in whiplash_dict_c_e.items():

        for event in events:

            if event['lag'] == lag:
                lag_dict[grid].append(event)

    lag_dicts[lag] = lag_dict

stress_full = (water_stress_da < ST20).astype(int)
nT = len(times)
whiplash_data = {}

for lag in range(1,7):
    
    print(f"Lag {lag}")
    
    lag_dict = lag_dicts[lag]
    
    arr = np.full((nT, nlat, nlon), np.nan)
    
    for (i,j), ev_list in lag_dict.items():
        
        for ev in ev_list:
            
            ev_start = times[ev["tc_index"]]
            ev_end   = times[ev["td_index"]]
            
            # map to full timeline
            t_start = times.searchsorted(ev_start)
            t_end   = times.searchsorted(ev_end) + 1
            
            for t in range(t_start, t_end):
                
                if np.isnan(arr[t,i,j]):
                    arr[t,i,j] = ev["score"]
                else:
                    arr[t,i,j] = max(arr[t,i,j], ev["score"])
    
    whiplash_data[lag] = arr

trigger_rates = {}

for lag in range(1,7):
    
    print(f"Trigger rate: lag {lag}")
    
    arr = whiplash_data[lag]
    whiplash_month = (arr > 0).astype(int)
    
    trigger = np.full((nlat,nlon), np.nan)
    
    for i in range(nlat):
        for j in range(nlon):
            
            X = whiplash_month[:,i,j]
            Y = stress_full[:,i,j]
            
            trigger[i,j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)
    
    trigger_rates[lag] = trigger

def trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=1000, seed=None):
    """
    X, Y : 1D binary arrays
    tau : lag
    delT : window length
    n_iter : number of Monte Carlo simulations
    Returns: observed rate, p-value
    """
    np.random.seed(seed)
    
    # observed trigger rate
    r_obs = ECA_trigger_grid(X, Y, deltaT=3, tau=0)
    
    # number of events
    NX = int(np.nansum(X))
    NY = int(np.nansum(Y))
    
    if NX == 0 or NY == 0:
        return r_obs, np.nan
    
    r_null = []
    T = len(X)
    
    for _ in range(n_iter):
        # randomly place X and Y events in time
        X_rand = np.zeros(T, dtype=int)
        Y_rand = np.zeros(T, dtype=int)
        
        X_idx = np.random.choice(T, NX, replace=False)
        Y_idx = np.random.choice(T, NY, replace=False)
        
        X_rand[X_idx] = 1
        Y_rand[Y_idx] = 1
        
        r_null.append(ECA_trigger_grid(X_rand, Y_rand, tau))
    
    r_null = np.array(r_null)
    
    # p-value: fraction of random trigger rates >= observed
    p_value = (r_null >= r_obs).sum() / n_iter
    
    return r_obs, p_value

p_values = {}

for lag in range(1,7):
    
    print(f"Significance: lag {lag}")
    
    arr = whiplash_data[lag]
    whiplash_month = (arr > 0).astype(int)
    
    p_arr = np.full((nlat,nlon), np.nan)
    
    for i in range(nlat):
        for j in range(nlon):
            
            X = whiplash_month[:,i,j]
            Y = stress_full[:,i,j]
            
            _, p = trigger_rate_significance(
                X, Y, tau=0, delT=3, n_iter=500
            )
            
            p_arr[i,j] = p
    
    p_values[lag] = p_arr
trigger_rate_significant = {}

for lag in range(1,7):
    
    tr_da = xr.DataArray(
        trigger_rates[lag],
        coords={'latitude': lat_values, 'longitude': lon_values},
        dims=('latitude','longitude')
    )
    
    p_da = xr.DataArray(
        p_values[lag],
        coords={'latitude': lat_values, 'longitude': lon_values},
        dims=('latitude','longitude')
    )
    
    trigger_rate_significant[lag] = tr_da.where(p_da < 0.05)


In [ ]:
import xarray as xr

koppen = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\RobustKGMaps.nc")
print(koppen)

koppen_da = koppen['MasterMap1']

import xarray as xr
import numpy as np

import numpy as np

koppen_vals = koppen_da.values

koppen_major = np.full_like(koppen_vals, '', dtype=object)

koppen_major[(koppen_vals >= 1) & (koppen_vals <= 3)] = 'A'
koppen_major[(koppen_vals >= 4) & (koppen_vals <= 7)] = 'B'
koppen_major[(koppen_vals >= 8) & (koppen_vals <= 16)] = 'C'
koppen_major[(koppen_vals >= 17) & (koppen_vals <= 28)] = 'D'
koppen_major[(koppen_vals >= 29) & (koppen_vals <= 30)] = 'E'

lags = range(1, 7)
climates = ['A','B','C','D','E']

import numpy as np
import pandas as pd

heatmap_data = []

for cls in climates:
    
    row = []
    mask = (koppen_major == cls)
    
    for lag in lags:
        
        da = tr_da.values
        
        # extract values in this climate
        vals = da[mask]
        
        # mean of significant values (NaNs ignored)
        mean_val = np.nanmean(vals)
        
        row.append(mean_val)
    
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(
    heatmap_data,
    index=climates,
    columns=lags
)

heatmap_df.index = ['Tropical','Arid','Temperate','Cold','Polar']
heatmap_df.columns = [f"Lag {l}" for l in lags]


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

koppen= xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\RobustKGMaps.nc")

kg = koppen["MasterMap1"]   # use this variable

In [ ]:
subtypes = koppen.attrs["KGSubtype"].split(";")

In [ ]:
ids = list(map(int, koppen.attrs["KGID"].split(";")))

In [ ]:
id_to_subtype = dict(zip(ids, subtypes))

In [ ]:
def subtype_to_major(subtype):
    if subtype.startswith("A"):
        return 1  # Tropical
    elif subtype.startswith("B"):
        return 2  # Arid
    elif subtype.startswith("C"):
        return 3  # Temperate
    elif subtype.startswith("D"):
        return 4  # Cold
    elif subtype.startswith("E"):
        return 5  # Polar
    else:
        return 0  # missing/undefined

In [ ]:
major_map = np.zeros_like(kg.values)

for k, subtype in id_to_subtype.items():
    major_class = subtype_to_major(subtype)
    major_map[kg.values == k] = major_class

In [ ]:
major_map

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs

# --- Global font settings ---
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['font.size'] = 12
mpl.rcParams['font.weight'] = 'bold'

fig = plt.figure(figsize=(12,6))
ax = plt.axes(projection=ccrs.Robinson())

# Colormap
cmap = mpl.colors.ListedColormap([
    "#2ca25f",  # A Tropical
    "#fdae61",  # B Arid
    "#4575b4",  # C Temperate
    "#74add1",  # D Cold
    "#f46d43"   # E Polar
])

img = ax.pcolormesh(
    koppen["lon"], koppen["lat"], major_map,
    transform=ccrs.PlateCarree(),
    cmap=cmap
)

ax.coastlines()

# --- Colorbar ---
cbar = plt.colorbar(img, orientation='horizontal', pad=0.05)
cbar.set_ticks([1,2,3,4,5])
cbar.set_ticklabels([
    "A: Tropical",
    "B: Arid",
    "C: Temperate",
    "D: Cold",
    "E: Polar"
])

# Make colorbar ticks bold + Times New Roman
for label in cbar.ax.get_xticklabels():
    label.set_fontname('Times New Roman')
    label.set_fontsize(12)
    label.set_fontweight('bold')

# --- Title ---
plt.title(
    "Global Climate Zones (Köppen–Geiger Classification)",
    fontsize=12,
    fontweight='bold',
    fontname='Times New Roman'
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# --- Global font settings ---
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['font.size'] = 14
mpl.rcParams['font.weight'] = 'bold'

fig = plt.figure(figsize=(12,6))
ax = plt.axes(projection=ccrs.Robinson())

# --- Add grey ocean background ---
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')
ax.add_feature(cfeature.LAND, facecolor='white', edgecolor='black')

# Colormap
cmap = mpl.colors.ListedColormap([
    "#2ca25f",  # A Tropical
    "#fdae61",  # B Arid
    "#4575b4",  # C Temperate
    "#74add1",  # D Cold
    "#f46d43"   # E Polar
])

img = ax.pcolormesh(
    koppen["lon"], koppen["lat"], major_map,
    transform=ccrs.PlateCarree(),
    cmap=cmap
)

ax.coastlines()

# --- Gridlines with lat/lon labels ---
gl = ax.gridlines(
    draw_labels=True,
    linewidth=0.5,
    color='gray',
    alpha=0.7,
    linestyle='--'
)

gl.top_labels = False
gl.right_labels = False

# Style grid labels
gl.xlabel_style = {'size': 12, 'weight': 'bold', 'family': 'Times New Roman'}
gl.ylabel_style = {'size': 12, 'weight': 'bold', 'family': 'Times New Roman'}

# --- Colorbar ---
cbar = plt.colorbar(img, orientation='horizontal', pad=0.05)
cbar.set_ticks([1,2,3,4,5])
cbar.set_ticklabels([
    "A: Tropical",
    "B: Arid",
    "C: Temperate",
    "D: Cold",
    "E: Polar"
])

for label in cbar.ax.get_xticklabels():
    label.set_fontname('Times New Roman')
    label.set_fontsize(12)
    label.set_fontweight('bold')

# --- Title ---
#plt.title(
    #"Global Climate Zones (Köppen–Geiger Classification)",
    #fontsize=16,
    #fontweight='bold',
    #fontname='Times New Roman'
#)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# --- Font settings ---
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['font.size'] = 12
mpl.rcParams['font.weight'] = 'bold'

fig = plt.figure(figsize=(12,6))
ax = plt.axes(projection=ccrs.Robinson())

# --- Mask 0 values (so they appear as grey) ---
masked_map = np.ma.masked_where(major_map == 0, major_map)

# --- Colormap ---
cmap = mpl.colors.ListedColormap([
    "#2ca25f",  # A Tropical
    "#fdae61",  # B Arid
    "#4575b4",  # C Temperate
    "#74add1",  # D Cold
    "#f46d43"   # E Polar
])

# Set masked values (0) to grey
cmap.set_bad(color='lightgrey')

# --- Plot ---
img = ax.pcolormesh(
    koppen["lon"], koppen["lat"], masked_map,
    transform=ccrs.PlateCarree(),
    cmap=cmap
)

ax.coastlines()

# --- Gridlines (more visible) ---
gl = ax.gridlines(
    draw_labels=True,
    linewidth=0.8,          # increased thickness
    color='black',          # darker color
    alpha=0.6,              # more visible
    linestyle='--'
)

gl.top_labels = False
gl.right_labels = False

gl.xlabel_style = {'size': 12, 'weight': 'bold', 'family': 'Times New Roman'}
gl.ylabel_style = {'size': 12, 'weight': 'bold', 'family': 'Times New Roman'}

# --- Colorbar ---
cbar = plt.colorbar(img, orientation='horizontal', pad=0.05)
cbar.set_ticks([1,2,3,4,5])
cbar.set_ticklabels([
    "A: Tropical",
    "B: Arid",
    "C: Temperate",
    "D: Cold",
    "E: Polar"
])

for label in cbar.ax.get_xticklabels():
    label.set_fontname('Times New Roman')
    label.set_fontsize(12)
    label.set_fontweight('bold')

# --- Title ---
#plt.title(
    #"Global Climate Zones (Köppen–Geiger Classification)",
    #fontsize=12,
    #fontweight='bold',
    #fontname='Times New Roman'
#)

plt.tight_layout()
plt.show()

# Whiplash Characteristics

## Whiplash Score

In [ ]:
max_score = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\int_max_score_1980_2019.nc")

In [ ]:
max_score_da = max_score['max_filtered_whiplash_score_whole']

In [ ]:
import json
from collections import defaultdict

with open(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\filtered_whiplash_events.json", "r") as f:
    loaded_dict3 = json.load(f)

In [ ]:
whiplash_dict_c_e = defaultdict(list, {eval(k): v for k, v in loaded_dict3.items()})

In [ ]:
scores = []

for grid, events in whiplash_dict_c_e.items():
    for event in events:
        scores.append(event['score'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# --- Global font settings ---
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14
rcParams['font.weight'] = 'bold'

plt.figure(figsize=(8,5))

sns.histplot(scores, bins=30, kde=True)

# Labels and title (explicitly bold to be safe)
plt.xlabel("Whiplash Score", fontsize=14, fontweight='bold')
plt.ylabel("Frequency", fontsize=14, fontweight='bold')
plt.title("Distribution of Whiplash Event Scores", fontsize=14, fontweight='bold')

# Tick formatting
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')

plt.grid(True, linestyle='--', alpha=0.5)
# ---- Save high resolution ----
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\distribution_whiplash_scores.tif",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
whiplash_dict_c_e

In [ ]:
import numpy as np

In [ ]:
average_score_map = np.full((nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():
    scores = [ev["score"] for ev in ev_list]
    average_score_map[i, j] = np.mean(scores)

In [ ]:
average_score_map

In [ ]:
average_score_map_da = xr.DataArray(
    average_score_map,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="avg_whiplash_score"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ---------------------------
# GLOBAL FONT SETTINGS
# ---------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14

# ---------------------------
# SETTINGS
# ---------------------------
vmin = 0
vmax = 0.5

ticks = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
tick_labels = ["0.0", "0.1", "0.2", "0.3", "0.4", "≥0.5"]

norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

# ✅ Colorblind-friendly, toned-down colormap
cmap = "viridis"

# ---------------------------
# FIGURE
# ---------------------------
fig = plt.figure(figsize=(10, 8))

ax1 = fig.add_axes([0.08, 0.1, 0.70, 0.8],
                   projection=ccrs.PlateCarree())

cbar_ax = fig.add_axes([0.82, 0.23, 0.02, 0.5])

# ---------------------------
# MAP BASE
# ---------------------------
ax1.set_extent([-180, 180, -90, 90])

ax1.coastlines(resolution='110m', linewidth=0.8)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=0)

# ---------------------------
# MAIN PLOT
# ---------------------------
Main_plot = average_score_map_da.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    add_labels=False,
    transform=ccrs.PlateCarree()
)

# ---------------------------
# GRID + TICKS (NO Lat/Lon text)
# ---------------------------
gl = ax1.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)

xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax1.set_xticks(xticks)
ax1.set_yticks(yticks)

ax1.set_xticklabels(
    [f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
    fontsize=14,
    fontweight='bold'
)

ax1.set_yticklabels(
    [f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
    fontsize=14,
    fontweight='bold'
)

ax1.tick_params(
    axis='both',
    direction='in',
    length=6,
    width=1.5
)

# ---------------------------
# BORDER (Cartopy-safe)
# ---------------------------
ax1.spines['geo'].set_linewidth(2)
ax1.spines['geo'].set_edgecolor('black')

# ---------------------------
# COLORBAR
# ---------------------------
cbar = plt.colorbar(
    Main_plot,
    cax=cbar_ax,
    orientation='vertical',
    ticks=ticks
)

cbar.ax.set_yticklabels(tick_labels)

cbar.ax.tick_params(
    labelsize=14,
    width=1.2,
    length=5,
    direction='in'
)

# Make ticks bold
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

# Colorbar label
cbar.set_label(
    'Whiplash Score',
    fontsize=14,
    fontweight='bold'
)

cbar.outline.set_linewidth(1.5)

# ---------------------------
# TITLE
# ---------------------------
ax1.set_title(
    'Average Whiplash Score (1980–2019)',
    fontsize=14,
    fontweight='bold'
)

plt.show()

## Maximum Duration

In [ ]:
max_duration = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\int_max_dur_1980_2019.nc")

In [ ]:
max_duration_da = max_duration['max_whiplash_duration']

In [ ]:
durations = []

for events in whiplash_dict_c_e.values():
    for e in events:
        duration = e['td_index'] - e['tc_index'] + 1
        durations.append(duration)

min(durations), max(durations)

In [ ]:
import numpy as np
durations = np.array(durations)

In [ ]:
import pandas as pd
bins = [3, 4, 7, 10, durations.max()]
labels = ['Short (4)', 'Moderate (5–7)', 'Long (8–10)', 'Very Long (>10)']

duration_class = pd.cut(durations, bins=bins, labels=labels)

class_counts_duration = duration_class.value_counts().sort_index()
class_percent_duration = (class_counts_duration / class_counts_duration.sum()) * 100

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))

colors = ['#009E73', '#56B4E9', '#E69F00', '#D55E00']

# Labels with percentage
labels = [f"{name}\n{value:.1f}%" for name, value in class_percent.items()]

wedges, texts = plt.pie(
    class_percent,
    labels=labels,
    startangle=90,
    colors=colors,
    labeldistance=1.15
)

# Donut hole
centre_circle = plt.Circle((0,0),0.65,fc='white')
plt.gca().add_artist(centre_circle)

# ✅ Add title inside donut
plt.text(
    0, 0,
    "Distribution of Whiplash\n Event Duration (months)",
    ha='center',
    va='center',
    fontsize=16,
    fontweight='bold'
)

plt.axis('equal')

# ---- Save high resolution ----
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\distribution_duration.tif",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
avg_duration = np.full((nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():
    
    durations = []
    
    for ev in ev_list:
        duration = ev["td_index"] - ev["tc_index"] + 1
        durations.append(duration)
    
    if len(durations) > 0:
        avg_duration[i, j] = np.mean(durations)

In [ ]:
avg_duration_da = xr.DataArray(
    avg_duration,
    coords={
        "lat": lat_values,
        "lon": lon_values
    },
    dims=["lat", "lon"],
    name="avg_whiplash_duration"
)

In [ ]:
avg_duration_da

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ---------------------------
# GLOBAL FONT SETTINGS
# ---------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14

# ---------------------------
# FIGURE (manual layout to avoid overlap)
# ---------------------------
fig = plt.figure(figsize=(10, 8))

ax1 = fig.add_axes([0.08, 0.1, 0.70, 0.8],
                   projection=ccrs.PlateCarree())

ax1.set_extent([-180, 180, -90, 90])

# ---------------------------
# NORMALIZATION + COLORMAP
# ---------------------------
norm = mcolors.TwoSlopeNorm(vmin=4, vcenter=8, vmax=10)

# ✅ Colorblind-friendly
cmap = "YlGnBu"

# ---------------------------
# MAIN PLOT
# ---------------------------
Main_plot = avg_duration_da.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_labels=False,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

# ---------------------------
# COAST + OCEAN
# ---------------------------
ax1.coastlines(resolution='110m', linewidth=1)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=0)

# ---------------------------
# SCATTER OVERLAY
# ---------------------------
lat_vals = avg_duration_da['lat'].values
lon_vals = avg_duration_da['lon'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
vals = avg_duration_da.values

mask = np.isfinite(vals)

ax1.scatter(
    lon2d[mask],
    lat2d[mask],
    c=vals[mask],
    cmap=cmap,
    norm=norm,
    s=1,
    edgecolors='none',
    transform=ccrs.PlateCarree(),
    zorder=5
)

# ---------------------------
# GRIDLINES
# ---------------------------
ax1.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)

# ---------------------------
# LAT-LON TICKS
# ---------------------------
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax1.set_xticks(xticks)
ax1.set_yticks(yticks)

ax1.set_xticklabels(
    [f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
    fontsize=14,
    fontweight='bold'
)

ax1.set_yticklabels(
    [f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
    fontsize=14,
    fontweight='bold'
)

ax1.tick_params(
    axis='both',
    direction='in',
    length=6,
    width=1.5
)

# ---------------------------
# BORDER
# ---------------------------
ax1.spines['geo'].set_linewidth(2)
ax1.spines['geo'].set_edgecolor('black')

# ---------------------------
# TITLE
# ---------------------------
ax1.set_title(
    'Average Whiplash Event Duration (1980–2019)',
    fontsize=14,
    fontweight='bold'
)

# ---------------------------
# COLORBAR (properly spaced)
# ---------------------------
cbar_ax = fig.add_axes([0.82, 0.23, 0.02, 0.5])

vmin = 4
vmax = 10
ticks = np.arange(vmin, vmax + 2, 2)

tick_labels = [str(t) for t in ticks]
tick_labels[-1] = "≥10"

cbar = plt.colorbar(
    Main_plot,
    cax=cbar_ax,
    orientation='vertical',
    ticks=ticks
)

cbar.ax.set_yticklabels(tick_labels)

# ✅ Bold + size 14 ticks
for label in cbar.ax.get_yticklabels():
    label.set_fontsize(14)
    label.set_fontweight('bold')

cbar.ax.tick_params(
    width=1.2,
    length=5,
    direction='in'
)

cbar.set_label(
    'Duration (months)',
    fontsize=14,
    fontweight='bold'
)

cbar.outline.set_linewidth(1.5)

plt.show()

## Decadal frequency

In [ ]:
import pandas as pd

rows = []

for (lat_i, lon_i), events in whiplash_dict_c_e.items():
    for e in events:
        rows.append({
            "lat": e["lat"],
            "lon": e["lon"],
            "year": pd.to_datetime(e["ta_time"]).year,
            "score": e["score"]
        })

whiplash_df = pd.DataFrame(rows)

In [ ]:
whiplash_df["decade"] = (whiplash_df["year"] // 10) * 10

In [ ]:
decadal_freq= (
    whiplash_df
    .groupby(["lat", "lon", "decade"])
    .size()
    .reset_index(name="count")
)

In [ ]:
avg_decadal_freq = (
    decadal_freq
    .groupby(["lat", "lon"])["count"]
    .mean()
    .reset_index(name="avg_decadal_count")
)

In [ ]:
import xarray as xr
import numpy as np

def df_to_full_grid(df, full_lats, full_lons, value_col="count"):

    grid = xr.DataArray(
        np.full((len(full_lats), len(full_lons)), np.nan),
        coords=[full_lats, full_lons],
        dims=["lat", "lon"]
    )

    for _, r in df.iterrows():
        grid.loc[dict(lat=r["lat"], lon=r["lon"])] = r[value_col]

    return grid


In [ ]:
avg_dec_freq_grid = df_to_full_grid(
    avg_decadal_freq,
    lat_values,
    lon_values,
    value_col="avg_decadal_count"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ---------------------------
# GLOBAL FONT SETTINGS
# ---------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14

# ---------------------------
# FIGURE
# ---------------------------
fig = plt.figure(figsize=(10, 8))

# Make main axis narrower (leave space on right)
ax1 = fig.add_axes([0.08, 0.1, 0.70, 0.8],
                   projection=ccrs.PlateCarree(central_longitude=0))

ax1.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

# ---------------------------
# NORMALIZATION + COLORMAP
# ---------------------------
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=1.5, vmax=3)

# ✅ Colorblind-friendly
cmap = "plasma"

# ---------------------------
# MAIN PLOT
# ---------------------------
Main_plot = avg_dec_freq_grid.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_labels=False,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

# ---------------------------
# COAST + OCEAN
# ---------------------------
ax1.coastlines(resolution='110m', linewidth=1)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=0)

# ---------------------------
# SCATTER OVERLAY
# ---------------------------
lat_vals = avg_dec_freq_grid['lat'].values
lon_vals = avg_dec_freq_grid['lon'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
vals = avg_dec_freq_grid.values

mask = np.isfinite(vals)

ax1.scatter(
    lon2d[mask],
    lat2d[mask],
    c=vals[mask],
    cmap=cmap,
    norm=norm,
    s=1,
    edgecolors='none',
    transform=ccrs.PlateCarree(),
    zorder=5
)

# ---------------------------
# GRIDLINES (NO LAT/LON TEXT)
# ---------------------------
ax1.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)

# ---------------------------
# TICKS
# ---------------------------
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax1.set_xticks(xticks)
ax1.set_yticks(yticks)

ax1.set_xticklabels(
    [f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
    fontsize=14,
    fontweight='bold'
)

ax1.set_yticklabels(
    [f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
    fontsize=14,
    fontweight='bold'
)

ax1.tick_params(
    axis='both',
    direction='in',
    length=6,
    width=1.5
)

# ---------------------------
# BORDER
# ---------------------------
ax1.spines['geo'].set_linewidth(2)
ax1.spines['geo'].set_edgecolor('black')

# ---------------------------
# TITLE
# ---------------------------
ax1.set_title(
    'Average Decadal Whiplash Frequency (1980–2019)',
    fontsize=14,
    fontweight='bold'
)

# ---------------------------
# COLORBAR
# ---------------------------
cbar_ax = fig.add_axes([0.82, 0.23, 0.02, 0.5])

vmin = 0
vmax = 3
ticks = np.arange(vmin, vmax + 1, 1)
tick_labels = [str(t) for t in ticks]
tick_labels[-1] = "≥3"

cbar = plt.colorbar(
    Main_plot,
    cax=cbar_ax,
    orientation='vertical',
    ticks=ticks
)

cbar.ax.set_yticklabels(tick_labels)

# Bold + size 14 ticks
for label in cbar.ax.get_yticklabels():
    label.set_fontsize(14)
    label.set_fontweight('bold')

cbar.ax.tick_params(
    width=1.2,
    length=5,
    direction='in'
)

cbar.set_label(
    'Frequency (events/decade)',
    fontsize=14,
    fontweight='bold'
)

cbar.outline.set_linewidth(1.5)

plt.show()

## Frequency

In [ ]:
max_freq= xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\int_max_freq_1980_2019.nc")

In [ ]:
max_freq

In [ ]:
max_freq_da = max_freq['__xarray_dataarray_variable__']

In [ ]:
import pandas as pd

rows = []

for (lat_i, lon_i), events in whiplash_dict_c_e.items():
    for e in events:
        rows.append({
            "lat": e["lat"],
            "lon": e["lon"],
            "year": pd.to_datetime(e["ta_time"]).year,
            "score": e["score"]
        })

whiplash_df_80 = pd.DataFrame(rows)

In [ ]:
whiplash_df_80["decade"] = (whiplash_df_80["year"] // 10) * 10

In [ ]:
grun=xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\GRUN\G-RUN_ENSEMBLE_MMM.nc")
# Rename coordinates
grun = grun.rename({"X": "lon", "Y": "lat"})

# Select time range 1980-01-01 to 2019-12-31
runoff= grun.sel(time=slice("1980-01-01", "2019-12-31"))

In [ ]:
nlat = grun.sizes['lat']
nlon = grun.sizes['lon']

lat_values = grun['lat'].values
lon_values = grun['lon'].values

In [ ]:
full_lats = lat_values
full_lons = lon_values

In [ ]:
decadal_freq_80 = (
    whiplash_df_80
    .groupby(["lat", "lon", "decade"])
    .size()
    .reset_index(name="count")
)

In [ ]:
bins = [0, 1, 2, 3, 5, 8]
labels = [
    'Very Low (1)',
    'Low (2)',
    'Moderate (3)',
    'High (4–5)',
    'Very High (≥6)'
]

decadal_freq_80['freq_class'] = pd.cut(
   decadal_freq_80['count'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

class_counts_freq = decadal_freq_80['freq_class'].value_counts().sort_index()
class_percent_freq = (class_counts_freq / class_counts_freq.sum()) * 100

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14

plt.figure(figsize=(8,5))
colors = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']

class_percent.plot(kind='barh', color=colors)

for i, v in enumerate(class_percent):
    if v < 1:
        plt.text(v + 0.2, i, f"{v:.2f}%", va='center', fontsize=12, fontweight='bold')
    else:
        plt.text(v + 0.5, i, f"{v:.1f}%", va='center')

plt.xlabel('Percentage of Grid Cells (%)', fontweight='bold')
plt.xlim(0, 100)
plt.ylabel('Frequency Class (Events/decade)', fontweight='bold')
plt.title('Spatial Distribution of Whiplash Event Frequency', fontweight='bold')

plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()

# ---- Save high resolution ----
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\spatial_distribution_frequency.tif",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

## IDF Characteristics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib import rcParams

rcParams["font.family"] = "Times New Roman"
rcParams["font.size"] = 12

fig = plt.figure(figsize=(18, 10), dpi=600)

gs = fig.add_gridspec(
    2, 3,
    height_ratios=[1, 0.85],
    wspace=0.35,
    hspace=0.25
)

# ---------------- MAP AXES ----------------
ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
ax3 = fig.add_subplot(gs[0, 2], projection=ccrs.PlateCarree())

# ---------------- DISTRIBUTION AXES ----------------
ax4 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1])
ax6 = fig.add_subplot(gs[1, 2])

# =====================================================
# (a) Maximum Whiplash Score
# =====================================================
norm1 = mcolors.TwoSlopeNorm(vmin=0, vcenter=0.25, vmax=0.5)

m1 = max_score_da.plot(
    ax=ax1,
    cmap="turbo",
    norm=norm1,
    add_colorbar=False,
    add_labels=False,
    transform=ccrs.PlateCarree()
)

ax1.set_title("(a) Maximum Whiplash Score", fontweight="bold")

# =====================================================
# (b) Maximum Whiplash Event Duration
# =====================================================
norm2 = mcolors.TwoSlopeNorm(vmin=0, vcenter=5, vmax=10)

m2 = max_duration_da.plot(
    ax=ax2,
    cmap="YlGnBu",
    norm=norm2,
    add_colorbar=False,
    add_labels=False,
    transform=ccrs.PlateCarree()
)

ax2.set_title("(b) Maximum Whiplash Event Duration", fontweight="bold")

# =====================================================
# (c) Maximum Whiplash Decadal Frequency
# =====================================================
norm3 = mcolors.TwoSlopeNorm(vmin=0, vcenter=1.5, vmax=3)

m3 = max_freq_da.plot(
    ax=ax3,
    cmap="plasma",
    norm=norm3,
    add_colorbar=False,
    add_labels=False,
    transform=ccrs.PlateCarree()
)

ax3.set_title("(c) Maximum Whiplash Decadal Frequency", fontweight="bold")

for ax in [ax1, ax2, ax3]:
    ax.set_extent([-180, 180, -90, 90])
    ax.coastlines(resolution="110m", linewidth=0.8)
    ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", zorder=0)

    ax.gridlines(
        draw_labels=False,
        linewidth=0.4,
        color="black",
        linestyle="--",
        alpha=0.4
    )

    xticks = np.arange(-180, 181, 90)
    yticks = np.arange(-90, 91, 45)

    ax.set_xticks(xticks)
    ax.set_yticks(yticks)

    ax.set_xticklabels(
        [f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
        fontsize=9,
        fontweight="bold"
    )

    ax.set_yticklabels(
        [f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
        fontsize=9,
        fontweight="bold"
    )

    ax.spines["geo"].set_linewidth(1.3)
    ax.spines["geo"].set_edgecolor("black")

cbar1 = fig.colorbar(m1, ax=ax1, orientation="horizontal", fraction=0.046, pad=0.08)
cbar1.set_label("Whiplash Score", fontweight="bold")

cbar2 = fig.colorbar(m2, ax=ax2, orientation="horizontal", fraction=0.046, pad=0.08)
cbar2.set_label("Duration (months)", fontweight="bold")

cbar3 = fig.colorbar(m3, ax=ax3, orientation="horizontal", fraction=0.046, pad=0.08)
cbar3.set_label("Frequency (events/decade)", fontweight="bold")

# =====================================================
# (d) Distribution of Whiplash Event Scores
# =====================================================
ax4.hist(scores, bins=30, edgecolor="black")
ax4.set_title("(d) Distribution of Whiplash Event Scores", fontweight="bold")
ax4.set_xlabel("Whiplash Score", fontweight="bold")
ax4.set_ylabel("Frequency", fontweight="bold")
ax4.grid(True, linestyle="--", alpha=0.5)

## =====================================================
# (e) Distribution of Whiplash Event Duration
# =====================================================

ax5.set_title(
    "(e) Distribution of Whiplash Event Duration",
    fontsize=14,
    fontweight="bold",
    pad=5
)

wedges, texts = ax5.pie(
    class_percent_duration,
    labels=None,
    startangle=90,
    radius=0.72,
    center=(0, -0.08),   # moves donut downward
    wedgeprops=dict(width=0.35, edgecolor="white")
)

# Put legend below instead of labels around donut
ax5.legend(
    wedges,
    [f"{name}: {value:.1f}%" for name, value in class_percent_duration.items()],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=2,
    frameon=False,
    fontsize=10
)

ax5.text(
    0, 0,
    "Duration\n(months)",
    ha="center",
    va="center",
    fontsize=11,
    fontweight="bold"
)

ax5.axis("equal")


# =====================================================
# (f) Distribution of Whiplash Event Frequency
# =====================================================
# IMPORTANT: use your frequency percentage variable here
# Rename your frequency class_percent as class_percent_frequency

class_percent_freq.plot(
    kind="barh",
    ax=ax6
)

for i, v in enumerate(class_percent_freq):
    ax6.text(
        v + 0.5,
        i,
        f"{v:.1f}%",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

ax6.set_title("(f) Distribution of Whiplash Event Frequency", fontweight="bold")
ax6.set_xlabel("Percentage of Grid Cells (%)", fontweight="bold")
ax6.set_ylabel("Frequency Class", fontweight="bold")
ax6.set_xlim(0, 100)
ax6.grid(axis="x", linestyle="--", alpha=0.5)

plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\combined_whiplash_6_panel.tif",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib import rcParams

# =====================================================
# GLOBAL SETTINGS
# =====================================================

rcParams["font.family"] = "Times New Roman"
rcParams["font.size"] = 12

fig = plt.figure(figsize=(12, 7.5), dpi=600)

# 2 rows × 4 columns
gs = fig.add_gridspec(
    2, 4,
    wspace=0.30,
    hspace=0.25
)

# =====================================================
# MAP AXES
# =====================================================

# Top row: two equally sized maps
ax1 = fig.add_subplot(
    gs[0, 0:2],
    projection=ccrs.PlateCarree()
)

ax2 = fig.add_subplot(
    gs[0, 2:4],
    projection=ccrs.PlateCarree()
)

# Bottom row: centered map
ax3 = fig.add_subplot(
    gs[1, 1:3],
    projection=ccrs.PlateCarree()
)

# =====================================================
# (a) AVERAGE WHIPLASH SCORE
# =====================================================

norm1 = mcolors.TwoSlopeNorm(
    vmin=0,
    vcenter=0.25,
    vmax=0.5
)

m1 = average_score_map_da.plot(
    ax=ax1,
    cmap="turbo",
    norm=norm1,
    add_colorbar=False,
    add_labels=False,
    transform=ccrs.PlateCarree()
)

ax1.set_title(
    "(a) Average Whiplash Score",
    fontsize=13,
    fontweight="bold"
)

# =====================================================
# (b) AVERAGE WHIPLASH EVENT DURATION
# =====================================================

norm2 = mcolors.TwoSlopeNorm(
    vmin=0,
    vcenter=5,
    vmax=10
)

m2 = avg_duration_da.plot(
    ax=ax2,
    cmap="YlGnBu",
    norm=norm2,
    add_colorbar=False,
    add_labels=False,
    transform=ccrs.PlateCarree()
)

ax2.set_title(
    "(b) Average Whiplash Event Duration",
    fontsize=13,
    fontweight="bold"
)

# =====================================================
# (c) AVERAGE DECADAL FREQUENCY
# =====================================================

norm3 = mcolors.TwoSlopeNorm(
    vmin=0,
    vcenter=1.5,
    vmax=3
)

m3 = avg_dec_freq_grid.plot(
    ax=ax3,
    cmap="plasma",
    norm=norm3,
    add_colorbar=False,
    add_labels=False,
    transform=ccrs.PlateCarree()
)

ax3.set_title(
    "(c) Average Whiplash Decadal Frequency",
    fontsize=13,
    fontweight="bold"
)

# =====================================================
# COMMON MAP FORMATTING
# =====================================================

for ax in [ax1, ax2, ax3]:

    ax.set_extent(
        [-180, 180, -90, 90],
        crs=ccrs.PlateCarree()
    )

    ax.coastlines(
        resolution="110m",
        linewidth=0.8
    )

    ax.add_feature(
        cfeature.OCEAN,
        facecolor="lightgrey",
        zorder=0
    )

    ax.gridlines(
        draw_labels=False,
        linewidth=0.4,
        color="black",
        linestyle="--",
        alpha=0.4
    )

    xticks = np.arange(-180, 181, 90)
    yticks = np.arange(-90, 91, 45)

    ax.set_xticks(
        xticks,
        crs=ccrs.PlateCarree()
    )

    ax.set_yticks(
        yticks,
        crs=ccrs.PlateCarree()
    )

    ax.set_xticklabels(
        [
            f"{abs(x)}°E" if x >= 0
            else f"{abs(x)}°W"
            for x in xticks
        ],
        fontsize=9,
        fontweight="bold"
    )

    ax.set_yticklabels(
        [
            f"{abs(y)}°N" if y >= 0
            else f"{abs(y)}°S"
            for y in yticks
        ],
        fontsize=9,
        fontweight="bold"
    )

    ax.tick_params(
        axis="both",
        direction="in",
        length=5,
        width=1.2
    )

    ax.spines["geo"].set_linewidth(1.3)
    ax.spines["geo"].set_edgecolor("black")

# =====================================================
# COLORBARS
# =====================================================

cbar1 = fig.colorbar(
    m1,
    ax=ax1,
    orientation="horizontal",
    fraction=0.046,
    pad=0.08
)

cbar1.set_label(
    "Whiplash Score",
    fontsize=11,
    fontweight="bold"
)

cbar2 = fig.colorbar(
    m2,
    ax=ax2,
    orientation="horizontal",
    fraction=0.046,
    pad=0.08
)

cbar2.set_label(
    "Duration (months)",
    fontsize=11,
    fontweight="bold"
)

cbar3 = fig.colorbar(
    m3,
    ax=ax3,
    orientation="horizontal",
    fraction=0.046,
    pad=0.08
)

cbar3.set_label(
    "Frequency (events/decade)",
    fontsize=11,
    fontweight="bold"
)

# Bold colorbar ticks
for cbar in [cbar1, cbar2, cbar3]:
    cbar.ax.tick_params(
        labelsize=10,
        width=1.1,
        direction="in"
    )

    for label in cbar.ax.get_xticklabels():
        label.set_fontweight("bold")

    cbar.outline.set_linewidth(1.3)

# =====================================================
# SAVE
# =====================================================

plt.savefig(
    r"D:\IITD\Doctoral Thesis\Work and Progress ppts\nature communications\Plots_final\Supplementary\combined_average_whiplash_3_panel.tif",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("Average score:")
print("Mean =", float(average_score_map_da.mean(skipna=True)))
print("Median =", float(average_score_map_da.median(skipna=True)))

print("\nAverage duration:")
print("Mean =", float(avg_duration_da.mean(skipna=True)))
print("Median =", float(avg_duration_da.median(skipna=True)))

print("\nAverage decadal frequency:")
print("Mean =", float(avg_dec_freq_grid.mean(skipna=True)))
print("Median =", float(avg_dec_freq_grid.median(skipna=True)))